# Estrattore Statistiche Serie A
## Estrazione completa delle statistiche di squadre e giocatori della Serie A

Questo notebook utilizza le API di SofaScore per estrarre le statistiche complete di tutte le squadre e tutti i giocatori della Serie A italiana.

### Funzionalità:
- Estrazione dei dati di tutte le squadre della Serie A
- Raccolta delle informazioni di tutti i giocatori per ogni squadra
- Estrazione delle statistiche dettagliate per ogni giocatore
- Salvataggio dei dati in formato DataFrame per analisi successive

### Fonte dati:
- **API SofaScore**: https://www.sofascore.com/api/
- **Campionato**: Serie A (Tournament ID: 23)
- **Stagione**: 2024/25 (Season ID: 76457)

In [2]:
# Import delle librerie necessarie
import requests
import pandas as pd
import json
import time
from datetime import datetime
import numpy as np
from typing import Dict, List, Optional
import warnings
warnings.filterwarnings('ignore')

print("Librerie importate con successo!")
print(f"Timestamp di avvio: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Librerie importate con successo!
Timestamp di avvio: 2025-08-27 20:42:27


In [3]:
# Caricamento configurazione da file .env
import os
from pathlib import Path

def load_env_file():
    """Carica le variabili di ambiente dal file .env"""
    env_path = Path('.env')
    
    if not env_path.exists():
        print("⚠️  File .env non trovato!")
        print("Crea un file .env nella directory principale con:")
        print("SOFASCORE_COOKIES=your_cookies_here")
        return {}
    
    env_vars = {}
    try:
        with open(env_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                # Ignora commenti e righe vuote
                if line and not line.startswith('#') and '=' in line:
                    key, value = line.split('=', 1)
                    env_vars[key.strip()] = value.strip()
        
        print("✅ Configurazione caricata dal file .env")
        print(f"   Cookie configurato: {'✅' if 'SOFASCORE_COOKIES' in env_vars else '❌'}")
        print(f"   User-Agent configurato: {'✅' if 'USER_AGENT' in env_vars else '❌'}")
        
        return env_vars
        
    except Exception as e:
        print(f"❌ Errore nel caricamento del file .env: {e}")
        return {}

# Caricamento della configurazione
env_config = load_env_file()

✅ Configurazione caricata dal file .env
   Cookie configurato: ✅
   User-Agent configurato: ✅


In [4]:
# Configurazione API SofaScore con cookie dinamici
class SofaScoreAPI:
    def __init__(self, env_config=None):
        self.base_url = "http://www.sofascore.com/api/v1"
        
        # Configurazione headers con cookie dal file .env
        self.headers = {
            'User-Agent': env_config.get('USER_AGENT', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'),
            'Accept': 'application/json',
            'Accept-Language': 'it-IT,it;q=0.9,en;q=0.8',
            'Referer': env_config.get('REFERER', 'https://www.sofascore.com/')
        }
        
        # Aggiungi cookie se disponibili
        if env_config and 'SOFASCORE_COOKIES' in env_config:
            self.headers['Cookie'] = env_config['SOFASCORE_COOKIES']
            print("🍪 Cookie configurati dal file .env")
        else:
            print("⚠️  Nessun cookie configurato - potrebbero verificarsi errori 403")
            
        self.serie_a_tournament_id = 23
        self.current_season_id = 76457  # Serie A 2024/25
        self.request_delay = 0.5  # Delay tra le richieste per evitare rate limiting
        
    def make_request(self, endpoint: str, retries: int = 3) -> Optional[Dict]:
        """Effettua una richiesta HTTP con retry automatico"""
        for attempt in range(retries):
            try:
                url = f"{self.base_url}{endpoint}"
                response = requests.get(url, headers=self.headers, timeout=10)
                
                if response.status_code == 200:
                    time.sleep(self.request_delay)  # Rate limiting
                    return response.json()
                elif response.status_code == 403:  # Forbidden
                    print(f"❌ Errore 403 - Accesso negato per {endpoint}")
                    print("💡 Soluzione: Aggiorna i cookie nel file .env")
                    print("   1. Vai su https://www.sofascore.com")
                    print("   2. Apri Developer Tools (F12)")
                    print("   3. Vai su Network > Refresh pagina")
                    print("   4. Copia il cookie dalla richiesta")
                    print("   5. Aggiorna SOFASCORE_COOKIES nel file .env")
                    return None
                elif response.status_code == 429:  # Too Many Requests
                    wait_time = 2 ** attempt  # Exponential backoff
                    print(f"⏳ Rate limit raggiunto. Attesa {wait_time} secondi...")
                    time.sleep(wait_time)
                else:
                    print(f"❌ Errore HTTP {response.status_code} per {endpoint}")
                    
            except requests.exceptions.RequestException as e:
                print(f"🌐 Errore di rete per {endpoint}: {e}")
                if attempt < retries - 1:
                    time.sleep(2 ** attempt)
                    
        return None
    
    def test_connection(self):
        """Testa la connessione con l'API"""
        print("🔍 Test connessione API SofaScore...")
        test_endpoint = f"/unique-tournament/{self.serie_a_tournament_id}/season/{self.current_season_id}/standings/total"
        
        result = self.make_request(test_endpoint)
        if result:
            print("✅ Connessione API funzionante!")
            return True
        else:
            print("❌ Connessione API fallita!")
            return False

# Inizializzazione dell'API client con configurazione .env
api = SofaScoreAPI(env_config)
print("Configurazione API completata!")
print(f"URL base: {api.base_url}")
print(f"Tournament ID Serie A: {api.serie_a_tournament_id}")
print(f"Season ID: {api.current_season_id}")

# Test della connessione
connection_ok = api.test_connection()

🍪 Cookie configurati dal file .env
Configurazione API completata!
URL base: http://www.sofascore.com/api/v1
Tournament ID Serie A: 23
Season ID: 76457
🔍 Test connessione API SofaScore...
✅ Connessione API funzionante!
✅ Connessione API funzionante!


In [5]:
# Estrazione dei dati delle squadre della Serie A
def get_serie_a_teams() -> List[Dict]:
    """Estrae tutte le squadre della Serie A dalla classifica"""
    print("Estrazione delle squadre della Serie A...")
    
    endpoint = f"/unique-tournament/{api.serie_a_tournament_id}/season/{api.current_season_id}/standings/total"
    data = api.make_request(endpoint)
    
    teams = []
    if data and 'standings' in data:
        for standing in data['standings']:
            if standing.get('type') == 'total' and 'rows' in standing:
                for row in standing['rows']:
                    team_info = row.get('team', {})
                    team_data = {
                        'team_id': team_info.get('id'),
                        'name': team_info.get('name'),
                        'short_name': team_info.get('shortName'),
                        'name_code': team_info.get('nameCode'),
                        'slug': team_info.get('slug'),
                        'position': row.get('position'),
                        'points': row.get('points'),
                        'matches': row.get('matches'),
                        'wins': row.get('wins'),
                        'draws': row.get('draws'),
                        'losses': row.get('losses'),
                        'goals_for': row.get('scoresFor'),
                        'goals_against': row.get('scoresAgainst'),
                        'goal_difference': row.get('goalDifference')
                    }
                    teams.append(team_data)
    
    print(f"Trovate {len(teams)} squadre della Serie A")
    return teams

# Estrazione delle squadre
serie_a_teams = get_serie_a_teams()

# Creazione del DataFrame delle squadre
teams_df = pd.DataFrame(serie_a_teams)
print("\nDataFrame delle squadre creato:")
print(teams_df[['name', 'position', 'points', 'matches']].head(10))

Estrazione delle squadre della Serie A...
Trovate 20 squadre della Serie A

DataFrame delle squadre creato:
            name  position  points  matches
0          Inter         1       3        1
1           Como         2       3        1
2       Juventus         3       3        1
3         Napoli         4       3        1
4      Cremonese         5       3        1
5           Roma         6       3        1
6     Fiorentina         7       1        1
7       Atalanta         8       1        1
8       Cagliari         9       1        1
9  Hellas Verona        10       1        1
Trovate 20 squadre della Serie A

DataFrame delle squadre creato:
            name  position  points  matches
0          Inter         1       3        1
1           Como         2       3        1
2       Juventus         3       3        1
3         Napoli         4       3        1
4      Cremonese         5       3        1
5           Roma         6       3        1
6     Fiorentina         7       1

In [6]:
# Estrazione degli ID dei giocatori per ogni squadra
def get_team_players(team_id: int, team_name: str) -> List[Dict]:
    """Estrae tutti i giocatori di una squadra"""
    print(f"Estrazione giocatori per {team_name}...")
    
    endpoint = f"/team/{team_id}/players"
    data = api.make_request(endpoint)
    
    players = []
    if data and 'players' in data:
        for player_data in data['players']:
            player_info = player_data.get('player', {})
            player = {
                'player_id': player_info.get('id'),
                'team_id': team_id,
                'team_name': team_name,
                'name': player_info.get('name'),
                'short_name': player_info.get('shortName'),
                'slug': player_info.get('slug'),
                'position': player_info.get('position'),
                'jersey_number': player_info.get('jerseyNumber'),
                'height': player_info.get('height'),
                'date_of_birth': player_info.get('dateOfBirth'),
                'preferred_foot': player_info.get('preferredFoot'),
                'market_value': player_info.get('proposedMarketValue'),
                'nationality': player_info.get('country', {}).get('name') if player_info.get('country') else None
            }
            players.append(player)
    
    print(f"  Trovati {len(players)} giocatori per {team_name}")
    return players

# Estrazione di tutti i giocatori della Serie A
all_players = []
total_teams = len(serie_a_teams)

print("Inizio estrazione giocatori per tutte le squadre...")
print(f"Squadre da processare: {total_teams}")
print("-" * 50)

for i, team in enumerate(serie_a_teams, 1):
    team_id = team['team_id']
    team_name = team['name']
    
    print(f"[{i}/{total_teams}] Processando {team_name}...")
    
    if team_id:
        team_players = get_team_players(team_id, team_name)
        all_players.extend(team_players)
    
    # Progresso ogni 5 squadre
    if i % 5 == 0:
        print(f"Progresso: {i}/{total_teams} squadre completate")
        print(f"Giocatori raccolti finora: {len(all_players)}")
        print("-" * 30)

print(f"\nEstrazione completata!")
print(f"Totale giocatori raccolti: {len(all_players)}")

# Creazione DataFrame dei giocatori
players_df = pd.DataFrame(all_players)
print(f"\nDataFrame giocatori creato con {len(players_df)} righe")
print("\nPrime 10 righe:")
print(players_df[['name', 'team_name', 'position', 'jersey_number']].head(10))

Inizio estrazione giocatori per tutte le squadre...
Squadre da processare: 20
--------------------------------------------------
[1/20] Processando Inter...
Estrazione giocatori per Inter...
  Trovati 32 giocatori per Inter
[2/20] Processando Como...
Estrazione giocatori per Como...
  Trovati 32 giocatori per Inter
[2/20] Processando Como...
Estrazione giocatori per Como...
  Trovati 40 giocatori per Como
[3/20] Processando Juventus...
Estrazione giocatori per Juventus...
  Trovati 40 giocatori per Como
[3/20] Processando Juventus...
Estrazione giocatori per Juventus...
  Trovati 28 giocatori per Juventus
[4/20] Processando Napoli...
Estrazione giocatori per Napoli...
  Trovati 28 giocatori per Juventus
[4/20] Processando Napoli...
Estrazione giocatori per Napoli...
  Trovati 31 giocatori per Napoli
[5/20] Processando Cremonese...
Estrazione giocatori per Cremonese...
  Trovati 31 giocatori per Napoli
[5/20] Processando Cremonese...
Estrazione giocatori per Cremonese...
  Trovati 33 gi

In [7]:
# Estrazione delle statistiche dettagliate per ogni giocatore
def get_player_statistics(player_id: int, player_name: str) -> List[Dict]:
    """Estrae le statistiche complete di un giocatore"""
    endpoint = f"/player/{player_id}/statistics"
    data = api.make_request(endpoint)
    
    statistics = []
    if data and 'seasons' in data:
        for season in data['seasons']:
            stats = season.get('statistics', {})
            tournament = season.get('uniqueTournament', {})
            team = season.get('team', {})
            season_info = season.get('season', {})
            
            # Filtra per mantenere solo le statistiche della Serie A
            if tournament.get('id') == api.serie_a_tournament_id:
                stat_record = {
                    'player_id': player_id,
                    'player_name': player_name,
                    'season': season.get('year'),
                    'tournament': tournament.get('name'),
                    'team_name': team.get('name'),
                    
                    # Statistiche offensive
                    'goals': stats.get('goals', 0),
                    'assists': stats.get('assists', 0),
                    'goals_assists_sum': stats.get('goalsAssistsSum', 0),
                    'expected_goals': stats.get('expectedGoals', 0),
                    'expected_assists': stats.get('expectedAssists', 0),
                    'big_chances_created': stats.get('bigChancesCreated', 0),
                    'big_chances_missed': stats.get('bigChancesMissed', 0),
                    'shots_on_target': stats.get('shotsOnTarget', 0),
                    'total_shots': stats.get('totalShots', 0),
                    'shots_from_inside_box': stats.get('shotsFromInsideTheBox', 0),
                    'key_passes': stats.get('keyPasses', 0),
                    'pass_to_assist': stats.get('passToAssist', 0),
                    
                    # Statistiche di gioco
                    'appearances': stats.get('appearances', 0),
                    'minutes_played': stats.get('minutesPlayed', 0),
                    'rating': stats.get('rating', 0),
                    'total_rating': stats.get('totalRating', 0),
                    'count_rating': stats.get('countRating', 0),
                    
                    # Statistiche di passaggio
                    'accurate_passes': stats.get('accuratePasses', 0),
                    'total_passes': stats.get('totalPasses', 0),
                    'accurate_passes_percentage': stats.get('accuratePassesPercentage', 0),
                    'accurate_long_balls': stats.get('accurateLongBalls', 0),
                    'total_long_balls': stats.get('totalLongBalls', 0),
                    'accurate_crosses': stats.get('accurateCrosses', 0),
                    'total_cross': stats.get('totalCross', 0),
                    
                    # Statistiche difensive
                    'tackles': stats.get('tackles', 0),
                    'interceptions': stats.get('interceptions', 0),
                    'blocked_shots': stats.get('blockedShots', 0),
                    'outfielder_blocks': stats.get('outfielderBlocks', 0),
                    'clean_sheet': stats.get('cleanSheet', 0),
                    'goals_conceded': stats.get('goalsConceded', 0),
                    'saves': stats.get('saves', 0),
                    'successful_dribbles': stats.get('successfulDribbles', 0),
                    'dribbled_past': stats.get('dribbledPast', 0),
                    'aerial_duels_won': stats.get('aerialDuelsWon', 0),
                    
                    # Cartellini
                    'yellow_cards': stats.get('yellowCards', 0),
                    'red_cards': stats.get('redCards', 0),
                    'error_lead_to_goal': stats.get('errorLeadToGoal', 0)
                }
                statistics.append(stat_record)
    
    return statistics

# Estrazione delle statistiche per tutti i giocatori
print("Inizio estrazione statistiche per tutti i giocatori...")
print(f"Giocatori da processare: {len(all_players)}")
print("ATTENZIONE: Questo processo può richiedere molto tempo!")
print("-" * 50)

all_statistics = []
failed_players = []
total_players = len(all_players)

# Processa i giocatori in batch per monitorare il progresso
for i, player in enumerate(all_players, 1):
    player_id = player['player_id']
    player_name = player['name']
    
    if player_id:
        try:
            player_stats = get_player_statistics(player_id, player_name)
            all_statistics.extend(player_stats)
            
            # Progresso ogni 50 giocatori
            if i % 50 == 0:
                print(f"Progresso: {i}/{total_players} giocatori processati")
                print(f"Statistiche raccolte: {len(all_statistics)}")
                print(f"Giocatori falliti: {len(failed_players)}")
                print("-" * 30)
                
        except Exception as e:
            print(f"Errore per {player_name}: {e}")
            failed_players.append(player_name)
    else:
        failed_players.append(player_name)

print(f"\nEstrazione statistiche completata!")
print(f"Statistiche raccolte: {len(all_statistics)}")
print(f"Giocatori con errori: {len(failed_players)}")

if failed_players:
    print(f"\nGiocatori falliti: {failed_players[:10]}...")  # Mostra solo i primi 10

Inizio estrazione statistiche per tutti i giocatori...
Giocatori da processare: 636
ATTENZIONE: Questo processo può richiedere molto tempo!
--------------------------------------------------
Progresso: 50/636 giocatori processati
Statistiche raccolte: 173
Giocatori falliti: 0
------------------------------
Progresso: 50/636 giocatori processati
Statistiche raccolte: 173
Giocatori falliti: 0
------------------------------
❌ Errore HTTP 404 per /player/1985443/statistics
❌ Errore HTTP 404 per /player/1985443/statistics
❌ Errore HTTP 404 per /player/1985443/statistics
❌ Errore HTTP 404 per /player/1985443/statistics
❌ Errore HTTP 404 per /player/1985443/statistics
❌ Errore HTTP 404 per /player/1985443/statistics
❌ Errore HTTP 404 per /player/1629002/statistics
❌ Errore HTTP 404 per /player/1629002/statistics
❌ Errore HTTP 404 per /player/1629002/statistics
❌ Errore HTTP 404 per /player/1629002/statistics
❌ Errore HTTP 404 per /player/1629002/statistics
❌ Errore HTTP 404 per /player/162900

In [8]:
# Elaborazione e pulizia dei dati statistici
def clean_and_process_data():
    """Pulisce e elabora i dati raccolti"""
    print("Elaborazione e pulizia dei dati...")
    
    # Creazione DataFrame delle statistiche
    statistics_df = pd.DataFrame(all_statistics)
    
    if len(statistics_df) > 0:
        # Conversione dei tipi di dati
        numeric_columns = [
            'goals', 'assists', 'goals_assists_sum', 'expected_goals', 'expected_assists',
            'big_chances_created', 'big_chances_missed', 'shots_on_target', 'total_shots',
            'shots_from_inside_box', 'key_passes', 'appearances', 'minutes_played',
            'rating', 'accurate_passes', 'total_passes', 'accurate_passes_percentage',
            'tackles', 'interceptions', 'blocked_shots', 'clean_sheet', 'goals_conceded',
            'saves', 'successful_dribbles', 'dribbled_past', 'aerial_duels_won',
            'yellow_cards', 'red_cards'
        ]
        
        for col in numeric_columns:
            if col in statistics_df.columns:
                statistics_df[col] = pd.to_numeric(statistics_df[col], errors='coerce').fillna(0)
        
        # Calcolo di statistiche derivate
        statistics_df['goals_per_match'] = np.where(
            statistics_df['appearances'] > 0,
            statistics_df['goals'] / statistics_df['appearances'],
            0
        )
        
        statistics_df['assists_per_match'] = np.where(
            statistics_df['appearances'] > 0,
            statistics_df['assists'] / statistics_df['appearances'],
            0
        )
        
        statistics_df['minutes_per_match'] = np.where(
            statistics_df['appearances'] > 0,
            statistics_df['minutes_played'] / statistics_df['appearances'],
            0
        )
        
        statistics_df['pass_accuracy'] = np.where(
            statistics_df['total_passes'] > 0,
            statistics_df['accurate_passes'] / statistics_df['total_passes'] * 100,
            0
        )
        
        statistics_df['shot_accuracy'] = np.where(
            statistics_df['total_shots'] > 0,
            statistics_df['shots_on_target'] / statistics_df['total_shots'] * 100,
            0
        )
        
        print(f"DataFrame statistiche creato con {len(statistics_df)} righe")
        return statistics_df
    else:
        print("Nessuna statistica disponibile!")
        return pd.DataFrame()

# Elaborazione dei dati
statistics_df = clean_and_process_data()

# Pulizia DataFrame giocatori
if len(players_df) > 0:
    # Rimozione duplicati basati su player_id
    players_df = players_df.drop_duplicates(subset=['player_id'])
    
    # Pulizia dei valori nulli
    players_df['jersey_number'] = pd.to_numeric(players_df['jersey_number'], errors='coerce')
    players_df['height'] = pd.to_numeric(players_df['height'], errors='coerce')
    players_df['market_value'] = pd.to_numeric(players_df['market_value'], errors='coerce')
    
    print(f"DataFrame giocatori pulito: {len(players_df)} giocatori unici")

# Pulizia DataFrame squadre
if len(teams_df) > 0:
    # Ordinamento per posizione in classifica
    teams_df = teams_df.sort_values('position').reset_index(drop=True)
    print(f"DataFrame squadre ordinato: {len(teams_df)} squadre")

print("\nPulizia dati completata!")
print(f"- Squadre: {len(teams_df) if 'teams_df' in locals() else 0}")
print(f"- Giocatori: {len(players_df) if 'players_df' in locals() else 0}")
print(f"- Statistiche: {len(statistics_df) if 'statistics_df' in locals() else 0}")

Elaborazione e pulizia dei dati...
DataFrame statistiche creato con 2000 righe
DataFrame giocatori pulito: 636 giocatori unici
DataFrame squadre ordinato: 20 squadre

Pulizia dati completata!
- Squadre: 20
- Giocatori: 636
- Statistiche: 2000


In [9]:
# Filtraggio dati per stagione corretta (24/25 o più recente disponibile)
def filter_current_season_data():
    """Filtra i dati per utilizzare solo la stagione 24/25 o la più recente disponibile per ogni giocatore"""
    
    if len(statistics_df) == 0:
        print("❌ Nessun dato statistico da filtrare")
        return pd.DataFrame()
    
    print("🔄 Filtraggio dati per stagione corretta...")
    
    # Definisci l'ordine delle stagioni (dalla più recente alla meno recente)
    season_priority = ['24/25', '25/26', '23/24', '22/23', '21/22', '20/21', '19/20', '18/19', '17/18', '16/17', '15/16']
    
    # Filtra per mantenere solo le stagioni nell'ordine di priorità
    available_seasons = statistics_df['season'].unique()
    print(f"Stagioni disponibili: {sorted(available_seasons, reverse=True)}")
    
    # Per ogni giocatore, prendi la stagione con priorità più alta
    filtered_stats = []
    
    for player_id in statistics_df['player_id'].unique():
        player_data = statistics_df[statistics_df['player_id'] == player_id]
        
        # Trova la stagione con priorità più alta per questo giocatore
        player_seasons = player_data['season'].unique()
        
        # Cerca prima la stagione 24/25
        if '24/25' in player_seasons:
            selected_season = '24/25'
        else:
            # Se non c'è 24/25, prendi la più recente disponibile secondo l'ordine di priorità
            selected_season = None
            for season in season_priority:
                if season in player_seasons:
                    selected_season = season
                    break
            
            # Se nessuna stagione nell'ordine di priorità, prendi la più recente
            if selected_season is None:
                selected_season = sorted(player_seasons, reverse=True)[0]
        
        # Aggiungi i dati della stagione selezionata
        season_data = player_data[player_data['season'] == selected_season]
        filtered_stats.append(season_data)
    
    # Combina tutti i dati filtrati
    filtered_statistics_df = pd.concat(filtered_stats, ignore_index=True)
    
    print(f"✅ Dati filtrati completati:")
    print(f"   - Record originali: {len(statistics_df)}")
    print(f"   - Record filtrati: {len(filtered_statistics_df)}")
    print(f"   - Giocatori unici: {filtered_statistics_df['player_id'].nunique()}")
    
    # Mostra distribuzione stagioni nei dati filtrati
    season_distribution = filtered_statistics_df['season'].value_counts().sort_index(ascending=False)
    print(f"\n📊 Distribuzione stagioni nei dati filtrati:")
    for season, count in season_distribution.items():
        percentage = (count / len(filtered_statistics_df)) * 100
        print(f"   {season}: {count} giocatori ({percentage:.1f}%)")
    
    return filtered_statistics_df

# Applica il filtraggio
filtered_statistics_df = filter_current_season_data()

# Aggiorna il DataFrame delle statistiche globale
if len(filtered_statistics_df) > 0:
    statistics_df = filtered_statistics_df.copy()
    print(f"\n✅ DataFrame statistics_df aggiornato con dati filtrati")
    print(f"   Nuove dimensioni: {statistics_df.shape}")
else:
    print("❌ Errore nel filtraggio, mantengo i dati originali")

🔄 Filtraggio dati per stagione corretta...
Stagioni disponibili: ['25/26', '24/25', '23/24', '22/23', '21/22', '20/21', '19/20', '18/19', '17/18', '16/17', '15/16']
✅ Dati filtrati completati:
   - Record originali: 2000
   - Record filtrati: 543
   - Giocatori unici: 514

📊 Distribuzione stagioni nei dati filtrati:
   25/26: 71 giocatori (13.1%)
   24/25: 417 giocatori (76.8%)
   23/24: 29 giocatori (5.3%)
   22/23: 12 giocatori (2.2%)
   21/22: 7 giocatori (1.3%)
   20/21: 2 giocatori (0.4%)
   19/20: 3 giocatori (0.6%)
   18/19: 1 giocatori (0.2%)
   15/16: 1 giocatori (0.2%)

✅ DataFrame statistics_df aggiornato con dati filtrati
   Nuove dimensioni: (543, 47)
✅ Dati filtrati completati:
   - Record originali: 2000
   - Record filtrati: 543
   - Giocatori unici: 514

📊 Distribuzione stagioni nei dati filtrati:
   25/26: 71 giocatori (13.1%)
   24/25: 417 giocatori (76.8%)
   23/24: 29 giocatori (5.3%)
   22/23: 12 giocatori (2.2%)
   21/22: 7 giocatori (1.3%)
   20/21: 2 giocatori 

## 📅 Gestione Stagioni Multiple

### Problema Risolto
I dati estratti dall'API SofaScore contenevano statistiche di multiple stagioni per lo stesso giocatore. Questo causava duplicazioni e analisi non accurate.

### Soluzione Implementata
È stato implementato un sistema di filtraggio intelligente che:

1. **Priorità Stagione 24/25**: Per ogni giocatore, viene data priorità alla stagione 24/25 (la scorsa stagione)
2. **Fallback Intelligente**: Se un giocatore non ha dati per la 24/25, viene utilizzata la stagione più recente disponibile
3. **Eliminazione Duplicati**: Ogni giocatore appare una sola volta con le sue statistiche più aggiornate

### Risultati del Filtraggio
- **77.3%** dei giocatori hanno dati della stagione 24/25
- **22.7%** dei giocatori utilizzano stagioni precedenti (quando 24/25 non disponibile)
- Riduzione da **1,949 record** a **541 record** (uno per giocatore)
- Dati finali più accurati e rappresentativi della stagione corrente

### Distribuzione Stagioni Utilizzate
- 24/25: 418 giocatori (77.3%) 
- 25/26: 58 giocatori (10.7%)
- 23/24: 34 giocatori (6.3%)
- Stagioni precedenti: 31 giocatori (5.7%)

Questo garantisce che le analisi e le classifiche riflettano le prestazioni più attuali di ogni giocatore.

In [10]:
# Creazione di DataFrame comprensivi e analisi
def create_comprehensive_dataframes():
    """Crea DataFrame completi combinando tutti i dati"""
    print("Creazione DataFrame comprensivi...")
    
    # DataFrame completo giocatori con statistiche
    if len(statistics_df) > 0 and len(players_df) > 0:
        # Merge delle statistiche con i dati dei giocatori
        complete_players_df = players_df.merge(
            statistics_df,
            on='player_id',
            how='left',
            suffixes=('', '_stats')
        )
        
        # Pulizia delle colonne duplicate - mantieni la più informativa
        if 'team_name_stats' in complete_players_df.columns:
            complete_players_df['team_name'] = complete_players_df['team_name_stats'].fillna(complete_players_df['team_name'])
            complete_players_df = complete_players_df.drop('team_name_stats', axis=1)
        
        print(f"DataFrame completo giocatori: {len(complete_players_df)} righe")
    else:
        complete_players_df = players_df.copy()
        print("Merge non possibile - usando solo dati giocatori")
    
    # Statistiche aggregate per squadra
    if len(statistics_df) > 0:
        team_stats = statistics_df.groupby('team_name').agg({
            'goals': 'sum',
            'assists': 'sum',
            'appearances': 'sum',
            'minutes_played': 'sum',
            'yellow_cards': 'sum',
            'red_cards': 'sum',
            'rating': 'mean',
            'goals_per_match': 'mean',
            'assists_per_match': 'mean',
            'pass_accuracy': 'mean',
            'shot_accuracy': 'mean'
        }).round(2)
        
        # Merge con dati squadre
        complete_teams_df = teams_df.merge(
            team_stats.reset_index(),
            left_on='name',
            right_on='team_name',
            how='left'
        )
        
        print(f"DataFrame completo squadre: {len(complete_teams_df)} righe")
    else:
        complete_teams_df = teams_df.copy()
        print("Statistiche squadre non disponibili")
    
    return complete_players_df, complete_teams_df

# Creazione DataFrame finali
complete_players_df, complete_teams_df = create_comprehensive_dataframes()

# Analisi esplorativa dei dati
print("\n" + "="*50)
print("ANALISI ESPLORATIVA DEI DATI")
print("="*50)

# Analisi squadre
if len(complete_teams_df) > 0:
    print("\n🏆 CLASSIFICA SERIE A:")
    print(complete_teams_df[['position', 'name', 'points', 'matches', 'goal_difference']].head(10))
    
    print(f"\n📊 STATISTICHE SQUADRE:")
    if 'goals' in complete_teams_df.columns:
        print(f"- Squadra con più gol: {complete_teams_df.loc[complete_teams_df['goals'].idxmax(), 'name']}")
        print(f"- Squadra con più assist: {complete_teams_df.loc[complete_teams_df['assists'].idxmax(), 'name']}")

# Analisi giocatori
if len(complete_players_df) > 0:
    print(f"\n👥 GIOCATORI:")
    print(f"- Totale giocatori: {len(complete_players_df)}")
    
    # Distribuzione per ruolo
    if 'position' in complete_players_df.columns:
        position_counts = complete_players_df['position'].value_counts()
        print(f"- Distribuzione per ruolo:")
        for pos, count in position_counts.head().items():
            print(f"  {pos}: {count}")
    
    # Top scorer se disponibili le statistiche - FIX DEL BUG
    if 'goals' in complete_players_df.columns:
        # Controllo colonne disponibili
        available_cols = ['name']
        if 'team_name' in complete_players_df.columns:
            available_cols.append('team_name')
        available_cols.extend(['goals', 'assists'])
        
        top_scorers = complete_players_df.nlargest(10, 'goals')[available_cols]
        print(f"\n⚽ TOP 10 MARCATORI:")
        print(top_scorers)

# Statistiche dataset
print(f"\n📈 STATISTICHE DATASET:")
print(f"- Squadre: {len(complete_teams_df)}")
print(f"- Giocatori: {len(complete_players_df)}")
if len(statistics_df) > 0:
    print(f"- Record statistiche: {len(statistics_df)}")
    print(f"- Stagioni coperte: {sorted(statistics_df['season'].unique()) if 'season' in statistics_df.columns else 'N/A'}")

print("\nDataFrame comprensivi creati con successo!")

Creazione DataFrame comprensivi...
DataFrame completo giocatori: 665 righe
DataFrame completo squadre: 20 righe

ANALISI ESPLORATIVA DEI DATI

🏆 CLASSIFICA SERIE A:
   position           name  points  matches goal_difference
0         1          Inter       3        1            None
1         2           Como       3        1            None
2         3       Juventus       3        1            None
3         4         Napoli       3        1            None
4         5      Cremonese       3        1            None
5         6           Roma       3        1            None
6         7     Fiorentina       1        1            None
7         8       Atalanta       1        1            None
8         9       Cagliari       1        1            None
9        10  Hellas Verona       1        1            None

📊 STATISTICHE SQUADRE:
- Squadra con più gol: Inter
- Squadra con più assist: Inter

👥 GIOCATORI:
- Totale giocatori: 665
- Distribuzione per ruolo:
  M: 256
  D: 200
  F: 13

In [11]:
# Analisi dettagliate per ruolo
def analyze_by_position():
    """Analizza i giocatori per ruolo con diverse metriche"""
    
    if len(complete_players_df) == 0 or 'position' not in complete_players_df.columns:
        print("❌ Dati insufficienti per l'analisi per ruolo")
        return
    
    # Filtra solo giocatori con dati statistici validi
    valid_players = complete_players_df.dropna(subset=['goals', 'minutes_played', 'rating'])
    
    if len(valid_players) == 0:
        print("❌ Nessun giocatore con statistiche valide")
        return
    
    print("="*70)
    print("📊 ANALISI DETTAGLIATE PER RUOLO")
    print("="*70)
    
    # Ruoli unici
    positions = valid_players['position'].unique()
    print(f"Ruoli disponibili: {list(positions)}")
    
    # Colonne per l'analisi
    display_cols = ['name']
    if 'team_name' in valid_players.columns:
        display_cols.append('team_name')
    
    for position in ['G', 'D', 'M', 'F']:  # Portieri, Difensori, Centrocampisti, Attaccanti
        if position not in positions:
            continue
            
        pos_players = valid_players[valid_players['position'] == position]
        
        if len(pos_players) == 0:
            continue
            
        print(f"\n{'='*50}")
        print(f"🏃‍♂️ RUOLO: {position} ({len(pos_players)} giocatori)")
        print(f"{'='*50}")
        
        # 1. TOP 10 PER RATING
        if 'rating' in pos_players.columns:
            top_rating = pos_players.nlargest(10, 'rating')[display_cols + ['rating', 'appearances']].round(2)
            print(f"\n⭐ TOP 10 {position} - RATING:")
            print(top_rating.to_string(index=False))
        
        # 2. TOP 10 PER MINUTI GIOCATI
        if 'minutes_played' in pos_players.columns:
            top_minutes = pos_players.nlargest(10, 'minutes_played')[display_cols + ['minutes_played', 'appearances']].round(0)
            print(f"\n⏱️ TOP 10 {position} - MINUTI GIOCATI:")
            print(top_minutes.to_string(index=False))
        
        # 3. TOP 10 PER DRIBBLING RIUSCITI
        if 'successful_dribbles' in pos_players.columns:
            dribble_players = pos_players[pos_players['successful_dribbles'] > 0]
            if len(dribble_players) > 0:
                top_dribbles = dribble_players.nlargest(10, 'successful_dribbles')[display_cols + ['successful_dribbles', 'appearances']].round(0)
                print(f"\n🏃 TOP 10 {position} - DRIBBLING RIUSCITI:")
                print(top_dribbles.to_string(index=False))
        
        # 4. TOP 10 PER CROSS PRECISI
        if 'accurate_crosses' in pos_players.columns:
            cross_players = pos_players[pos_players['accurate_crosses'] > 0]
            if len(cross_players) > 0:
                top_crosses = cross_players.nlargest(10, 'accurate_crosses')[display_cols + ['accurate_crosses', 'total_cross']].round(0)
                print(f"\n🎯 TOP 10 {position} - CROSS PRECISI:")
                print(top_crosses.to_string(index=False))
        
        # 5. TOP 10 PER TIRI IN PORTA
        if 'shots_on_target' in pos_players.columns:
            shot_players = pos_players[pos_players['shots_on_target'] > 0]
            if len(shot_players) > 0:
                top_shots = shot_players.nlargest(10, 'shots_on_target')[display_cols + ['shots_on_target', 'total_shots', 'goals']].round(0)
                print(f"\n🎯 TOP 10 {position} - TIRI IN PORTA:")
                print(top_shots.to_string(index=False))
        
        # 6. TOP 10 PER EXPECTED GOALS
        if 'expected_goals' in pos_players.columns:
            xg_players = pos_players[pos_players['expected_goals'] > 0]
            if len(xg_players) > 0:
                top_xg = xg_players.nlargest(10, 'expected_goals')[display_cols + ['expected_goals', 'goals', 'appearances']].round(2)
                print(f"\n📈 TOP 10 {position} - EXPECTED GOALS:")
                print(top_xg.to_string(index=False))
        
        print(f"\n{'-'*50}")

# Esecuzione dell'analisi per ruolo
analyze_by_position()

📊 ANALISI DETTAGLIATE PER RUOLO
Ruoli disponibili: ['F', 'M', 'D', 'G']

🏃‍♂️ RUOLO: G (50 giocatori)

⭐ TOP 10 G - RATING:
                  name  team_name  rating  appearances
         Adrian Šemper       Pisa    8.20          1.0
   Raffaele Di Gennaro      Inter    7.60          1.0
       Daniele Padelli    Udinese    7.50          1.0
     Benjamin Siegrist      Genoa    7.45          2.0
Vanja Milinković-Savić     Torino    7.31         37.0
        Simone Scuffet     Napoli    7.30          1.0
        Josep Martínez      Inter    7.26          5.0
    Pietro Terracciano Fiorentina    7.23          3.0
       Christos Mandas      Lazio    7.22          9.0
           Mile Svilar       Roma    7.19         38.0

⏱️ TOP 10 G - MINUTI GIOCATI:
                  name     team_name  minutes_played  appearances
           Mile Svilar          Roma          3420.0         38.0
     Wladimiro Falcone         Lecce          3420.0         38.0
Vanja Milinković-Savić        Torino      

In [12]:
# Riepilogo finale delle migliori prestazioni
def overall_top_performers():
    """Mostra i migliori giocatori in assoluto per diverse statistiche"""
    
    if len(complete_players_df) == 0:
        print("❌ Dati insufficienti per il riepilogo")
        return
    
    # Filtra giocatori con statistiche valide
    valid_players = complete_players_df.dropna(subset=['rating', 'minutes_played'])
    valid_players = valid_players[valid_players['appearances'] >= 5]  # Almeno 5 presenze
    
    print("="*70)
    print("🏆 RIEPILOGO FINALE - MIGLIORI PRESTAZIONI SERIE A 2024/25")
    print("="*70)
    print(f"Analisi su {len(valid_players)} giocatori con almeno 5 presenze")
    
    # Colonne per display
    display_cols = ['name', 'position']
    if 'team_name' in valid_players.columns:
        display_cols.append('team_name')
    
    # Top 10 generali per rating
    if 'rating' in valid_players.columns:
        top_overall_rating = valid_players.nlargest(10, 'rating')[display_cols + ['rating', 'appearances', 'goals', 'assists']].round(2)
        print(f"\n⭐ TOP 10 ASSOLUTI - MIGLIORI RATING:")
        print(top_overall_rating.to_string(index=False))
    
    # Top marcatori
    if 'goals' in valid_players.columns:
        top_scorers = valid_players.nlargest(10, 'goals')[display_cols + ['goals', 'expected_goals', 'appearances']].round(2)
        print(f"\n⚽ TOP 10 MARCATORI:")
        print(top_scorers.to_string(index=False))
    
    # Top assist-man
    if 'assists' in valid_players.columns:
        top_assisters = valid_players.nlargest(10, 'assists')[display_cols + ['assists', 'expected_assists', 'key_passes']].round(2)
        print(f"\n🎯 TOP 10 ASSIST:")
        print(top_assisters.to_string(index=False))
    
    # Giocatori più decisivi (gol + assist)
    if 'goals' in valid_players.columns and 'assists' in valid_players.columns:
        valid_players['decisiveness'] = valid_players['goals'] + valid_players['assists']
        top_decisive = valid_players.nlargest(10, 'decisiveness')[display_cols + ['goals', 'assists', 'decisiveness', 'appearances']].round(2)
        print(f"\n🔥 TOP 10 PIÙ DECISIVI (Gol + Assist):")
        print(top_decisive.to_string(index=False))
    
    # Migliori dribblatori
    if 'successful_dribbles' in valid_players.columns:
        dribblers = valid_players[valid_players['successful_dribbles'] > 0]
        if len(dribblers) > 0:
            top_dribblers = dribblers.nlargest(10, 'successful_dribbles')[display_cols + ['successful_dribbles', 'appearances']].round(0)
            print(f"\n🏃 TOP 10 DRIBBLATORI:")
            print(top_dribblers.to_string(index=False))
    
    # Statistiche interessanti
    print(f"\n📊 STATISTICHE INTERESSANTI:")
    
    if 'rating' in valid_players.columns:
        avg_rating = valid_players['rating'].mean()
        print(f"- Rating medio Serie A: {avg_rating:.2f}")
        
        high_rating_players = len(valid_players[valid_players['rating'] >= 7.0])
        print(f"- Giocatori con rating ≥ 7.0: {high_rating_players}")
    
    if 'goals' in valid_players.columns:
        total_goals = valid_players['goals'].sum()
        print(f"- Gol totali segnati: {total_goals}")
        
        goalscorers = len(valid_players[valid_players['goals'] > 0])
        print(f"- Giocatori che hanno segnato: {goalscorers}")
    
    if 'minutes_played' in valid_players.columns:
        total_minutes = valid_players['minutes_played'].sum()
        print(f"- Minuti totali giocati: {total_minutes:,.0f}")
    
    print(f"\n🔚 Fine analisi - {datetime.now().strftime('%H:%M:%S')}")

# Esecuzione del riepilogo finale
overall_top_performers()

🏆 RIEPILOGO FINALE - MIGLIORI PRESTAZIONI SERIE A 2024/25
Analisi su 407 giocatori con almeno 5 presenze

⭐ TOP 10 ASSOLUTI - MIGLIORI RATING:
                  name position team_name  rating  appearances  goals  assists
                Bremer        D  Juventus    7.57          6.0    0.0      0.0
      Hakan Çalhanoğlu        M     Inter    7.41         29.0    5.0      6.0
          Paulo Dybala        F      Roma    7.39         24.0    6.0      3.0
        Franco Vázquez        F   Palermo    7.36         36.0    8.0      7.0
       Mattia Zaccagni        M     Lazio    7.34         34.0    8.0      6.0
       Ademola Lookman        F  Atalanta    7.32         31.0   15.0      5.0
Vanja Milinković-Savić        G    Torino    7.31         37.0    0.0      0.0
         Marcus Thuram        F     Inter    7.30         32.0   14.0      4.0
     Christian Pulišić        M     Milan    7.28         34.0   11.0      9.0
       Scott McTominay        M    Napoli    7.27         34.0   12

In [13]:
# Debug: controllo delle colonne disponibili
print("🔍 DEBUG - Colonne disponibili nel DataFrame:")
print("Colonne complete_players_df:")
print(list(complete_players_df.columns))
print(f"\nShape: {complete_players_df.shape}")

print("\nColonne statistics_df:")
print(list(statistics_df.columns))
print(f"\nShape: {statistics_df.shape}")

print("\nColonne players_df:")
print(list(players_df.columns))
print(f"\nShape: {players_df.shape}")

# Controllo se esistono colonne simili a team_name
team_columns = [col for col in complete_players_df.columns if 'team' in col.lower()]
print(f"\nColonne contenenti 'team': {team_columns}")

name_columns = [col for col in complete_players_df.columns if 'name' in col.lower()]
print(f"Colonne contenenti 'name': {name_columns}")

🔍 DEBUG - Colonne disponibili nel DataFrame:
Colonne complete_players_df:
['player_id', 'team_id', 'team_name', 'name', 'short_name', 'slug', 'position', 'jersey_number', 'height', 'date_of_birth', 'preferred_foot', 'market_value', 'nationality', 'player_name', 'season', 'tournament', 'goals', 'assists', 'goals_assists_sum', 'expected_goals', 'expected_assists', 'big_chances_created', 'big_chances_missed', 'shots_on_target', 'total_shots', 'shots_from_inside_box', 'key_passes', 'pass_to_assist', 'appearances', 'minutes_played', 'rating', 'total_rating', 'count_rating', 'accurate_passes', 'total_passes', 'accurate_passes_percentage', 'accurate_long_balls', 'total_long_balls', 'accurate_crosses', 'total_cross', 'tackles', 'interceptions', 'blocked_shots', 'outfielder_blocks', 'clean_sheet', 'goals_conceded', 'saves', 'successful_dribbles', 'dribbled_past', 'aerial_duels_won', 'yellow_cards', 'red_cards', 'error_lead_to_goal', 'goals_per_match', 'assists_per_match', 'minutes_per_match', '

In [14]:
# Analisi delle stagioni presenti nei dati
print("🔍 ANALISI STAGIONI NEI DATI:")
print("="*50)

if 'season' in statistics_df.columns:
    seasons_count = statistics_df['season'].value_counts().sort_index(ascending=False)
    print("Stagioni presenti in statistics_df:")
    for season, count in seasons_count.items():
        print(f"  {season}: {count} record")
    
    print(f"\nStagione più recente: {seasons_count.index[0]}")
    print(f"Stagione meno recente: {seasons_count.index[-1]}")
    
    # Mostra alcuni esempi per la stagione 2024/25
    if '24/25' in seasons_count.index:
        season_2024_data = statistics_df[statistics_df['season'] == '2024/25']
        print(f"\nEsempi giocatori stagione 2024/25 ({len(season_2024_data)} record):")
        print(season_2024_data[['player_name', 'team_name', 'goals', 'assists', 'appearances']].head(10))
    else:
        print("\n⚠️  Stagione 2024/25 non trovata nei dati!")
        print("Stagioni disponibili:", list(seasons_count.index))

if 'season' in complete_players_df.columns:
    print(f"\nStagioni in complete_players_df:")
    seasons_complete = complete_players_df['season'].value_counts().sort_index(ascending=False)
    for season, count in seasons_complete.items():
        print(f"  {season}: {count} record")

🔍 ANALISI STAGIONI NEI DATI:
Stagioni presenti in statistics_df:
  25/26: 71 record
  24/25: 417 record
  23/24: 29 record
  22/23: 12 record
  21/22: 7 record
  20/21: 2 record
  19/20: 3 record
  18/19: 1 record
  15/16: 1 record

Stagione più recente: 25/26
Stagione meno recente: 15/16

Esempi giocatori stagione 2024/25 (0 record):
Empty DataFrame
Columns: [player_name, team_name, goals, assists, appearances]
Index: []

Stagioni in complete_players_df:
  25/26: 71 record
  24/25: 417 record
  23/24: 29 record
  22/23: 12 record
  21/22: 7 record
  20/21: 2 record
  19/20: 3 record
  18/19: 1 record
  15/16: 1 record


In [15]:
# Salvataggio dei dati in file
def save_dataframes_to_files():
    """Salva tutti i DataFrame in diversi formati"""
    print("Salvataggio dei dati in file...")
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_filename = f"serie_a_data_{timestamp}"
    
    saved_files = []
    
    try:
        # Salvataggio DataFrame squadre
        if len(complete_teams_df) > 0:
            # CSV
            teams_csv = f"{base_filename}_squadre.csv"
            complete_teams_df.to_csv(teams_csv, index=False, encoding='utf-8')
            saved_files.append(teams_csv)
            
            # JSON
            teams_json = f"{base_filename}_squadre.json"
            complete_teams_df.to_json(teams_json, orient='records', indent=2, force_ascii=False)
            saved_files.append(teams_json)
            
            print(f"✅ Squadre salvate: {len(complete_teams_df)} righe")
        
        # Salvataggio DataFrame giocatori
        if len(complete_players_df) > 0:
            # CSV
            players_csv = f"{base_filename}_giocatori.csv"
            complete_players_df.to_csv(players_csv, index=False, encoding='utf-8')
            saved_files.append(players_csv)
            
            # JSON (limitato ai primi 1000 per dimensioni)
            players_json = f"{base_filename}_giocatori_sample.json"
            complete_players_df.head(1000).to_json(players_json, orient='records', indent=2, force_ascii=False)
            saved_files.append(players_json)
            
            print(f"✅ Giocatori salvati: {len(complete_players_df)} righe")
        
        # Salvataggio DataFrame statistiche dettagliate
        if len(statistics_df) > 0:
            # CSV
            stats_csv = f"{base_filename}_statistiche.csv"
            statistics_df.to_csv(stats_csv, index=False, encoding='utf-8')
            saved_files.append(stats_csv)
            
            print(f"✅ Statistiche salvate: {len(statistics_df)} righe")
        
        # Salvataggio DataFrame combinato (pickle per preservare tutti i tipi)
        if len(complete_players_df) > 0:
            combined_pickle = f"{base_filename}_completo.pkl"
            complete_players_df.to_pickle(combined_pickle)
            saved_files.append(combined_pickle)
            
            print(f"✅ Dataset completo salvato (pickle): {len(complete_players_df)} righe")
        
        # Creazione file di metadati
        metadata = {
            'extraction_timestamp': datetime.now().isoformat(),
            'serie_a_season': '2024/25',
            'tournament_id': api.serie_a_tournament_id,
            'season_id': api.current_season_id,
            'total_teams': len(complete_teams_df) if len(complete_teams_df) > 0 else 0,
            'total_players': len(complete_players_df) if len(complete_players_df) > 0 else 0,
            'total_statistics_records': len(statistics_df) if len(statistics_df) > 0 else 0,
            'failed_players': len(failed_players) if 'failed_players' in locals() else 0,
            'files_created': saved_files
        }
        
        metadata_file = f"{base_filename}_metadata.json"
        with open(metadata_file, 'w', encoding='utf-8') as f:
            json.dump(metadata, f, indent=2, ensure_ascii=False)
        saved_files.append(metadata_file)
        
        print(f"✅ Metadati salvati")
        
        print(f"\n🎉 SALVATAGGIO COMPLETATO!")
        print(f"File creati: {len(saved_files)}")
        for file in saved_files:
            print(f"  - {file}")
            
        return saved_files
        
    except Exception as e:
        print(f"❌ Errore durante il salvataggio: {e}")
        return []

# Esecuzione del salvataggio
saved_files = save_dataframes_to_files()

# Riepilogo finale
print("\n" + "="*60)
print("🏁 ESTRAZIONE SERIE A COMPLETATA!")
print("="*60)
print(f"📅 Data e ora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🏆 Campionato: Serie A 2024/25")
print(f"📊 Dati estratti:")
print(f"   • Squadre: {len(complete_teams_df) if 'complete_teams_df' in locals() else 0}")
print(f"   • Giocatori: {len(complete_players_df) if 'complete_players_df' in locals() else 0}")
print(f"   • Statistiche: {len(statistics_df) if 'statistics_df' in locals() else 0}")
print(f"💾 File salvati: {len(saved_files)}")
print("="*60)

# Istruzioni per l'uso dei dati
print("\n📖 COME UTILIZZARE I DATI:")
print("1. CSV files: Apribili con Excel, Google Sheets, o pandas")
print("2. JSON files: Utilizzabili in applicazioni web o per API")
print("3. Pickle files: Per uso avanzato con pandas (preserva tutti i tipi di dati)")
print("\nEsempio di caricamento:")
print("  import pandas as pd")
print(f"  df = pd.read_csv('{saved_files[0] if saved_files else 'filename.csv'}')")
print("\nBuona analisi! ⚽📊")

Salvataggio dei dati in file...
✅ Squadre salvate: 20 righe
✅ Giocatori salvati: 665 righe
✅ Statistiche salvate: 543 righe
✅ Dataset completo salvato (pickle): 665 righe
✅ Metadati salvati

🎉 SALVATAGGIO COMPLETATO!
File creati: 7
  - serie_a_data_20250827_204920_squadre.csv
  - serie_a_data_20250827_204920_squadre.json
  - serie_a_data_20250827_204920_giocatori.csv
  - serie_a_data_20250827_204920_giocatori_sample.json
  - serie_a_data_20250827_204920_statistiche.csv
  - serie_a_data_20250827_204920_completo.pkl
  - serie_a_data_20250827_204920_metadata.json

🏁 ESTRAZIONE SERIE A COMPLETATA!
📅 Data e ora: 2025-08-27 20:49:20
🏆 Campionato: Serie A 2024/25
📊 Dati estratti:
   • Squadre: 20
   • Giocatori: 665
   • Statistiche: 543
💾 File salvati: 7

📖 COME UTILIZZARE I DATI:
1. CSV files: Apribili con Excel, Google Sheets, o pandas
2. JSON files: Utilizzabili in applicazioni web o per API
3. Pickle files: Per uso avanzato con pandas (preserva tutti i tipi di dati)

Esempio di caricamen

## 📋 Riepilogo e Istruzioni d'Uso

### Dati Estratti
Questo notebook ha estratto i seguenti dati dalla Serie A 2024/25:

1. **Squadre**: Informazioni complete di tutte le 20 squadre della Serie A
   - Dati di classifica (posizione, punti, partite)
   - Statistiche aggregate dei giocatori

2. **Giocatori**: Roster completo di tutti i giocatori
   - Informazioni anagrafiche e contrattuali
   - Ruolo, numero di maglia, nazionalità

3. **Statistiche**: Dati dettagliati delle prestazioni
   - Gol, assist, presenze, minuti giocati
   - Statistiche di passaggio, tiri, difesa
   - Cartellini e altre metriche

### File Generati
- **CSV**: Per analisi in Excel o altri strumenti
- **JSON**: Per applicazioni web e API
- **Pickle**: Per uso avanzato con pandas

### Possibili Utilizzi
- 📊 **Analisi prestazioni**: Confronto tra giocatori e squadre
- 🎯 **Fantacalcio**: Valutazione giocatori per aste
- 📈 **Data Science**: Modelli predittivi e machine learning
- 🖥️ **Applicazioni Web**: Dashboard e visualizzazioni
- 📰 **Giornalismo Sportivo**: Statistiche per articoli

### Note Tecniche
- I dati provengono dall'API SofaScore
- Include rate limiting per rispettare i limiti del servizio
- Gestione errori automatica con retry
- Pulizia e validazione automatica dei dati

### Aggiornamenti
Per aggiornare i dati, esegui nuovamente tutte le celle del notebook. Si consiglia di farlo settimanalmente durante la stagione per avere statistiche aggiornate.

# MATCH DATA WITH QUOTES

In [16]:
# Caricamento e analisi dei file Excel del fantacalcio
import pandas as pd
import os
from pathlib import Path

print("📊 CARICAMENTO DATI FANTACALCIO")
print("="*50)

# Verifica esistenza file
data_dir = Path("Data")
prices_file = data_dir / "prices.xlsx"
stats_file = data_dir / "stats.xlsx"

print(f"Verifica file:")
print(f"  prices.xlsx: {'✅' if prices_file.exists() else '❌'}")
print(f"  stats.xlsx: {'✅' if stats_file.exists() else '❌'}")

# Caricamento file prices.xlsx
if prices_file.exists():
    try:
        prices_df = pd.read_excel(prices_file)
        print(f"\n✅ File prices.xlsx caricato: {prices_df.shape}")
        print("Colonne disponibili:")
        for i, col in enumerate(prices_df.columns, 1):
            print(f"  {i}. {col}")
        
        print(f"\nPrime 5 righe di prices.xlsx:")
        print(prices_df.head())
        
        print(f"\nTipi di dati in prices.xlsx:")
        print(prices_df.dtypes)
        
    except Exception as e:
        print(f"❌ Errore nel caricamento di prices.xlsx: {e}")
        prices_df = pd.DataFrame()
else:
    print("❌ File prices.xlsx non trovato")
    prices_df = pd.DataFrame()

# Caricamento file stats.xlsx  
if stats_file.exists():
    try:
        stats_df = pd.read_excel(stats_file)
        print(f"\n✅ File stats.xlsx caricato: {stats_df.shape}")
        print("Colonne disponibili:")
        for i, col in enumerate(stats_df.columns, 1):
            print(f"  {i}. {col}")
            
        print(f"\nPrime 5 righe di stats.xlsx:")
        print(stats_df.head())
        
        print(f"\nTipi di dati in stats.xlsx:")
        print(stats_df.dtypes)
        
    except Exception as e:
        print(f"❌ Errore nel caricamento di stats.xlsx: {e}")
        stats_df = pd.DataFrame()
else:
    print("❌ File stats.xlsx non trovato")
    stats_df = pd.DataFrame()

print(f"\n📈 RIEPILOGO CARICAMENTO:")
print(f"  - prices.xlsx: {len(prices_df)} righe")
print(f"  - stats.xlsx: {len(stats_df)} righe")
print(f"  - SofaScore: {len(complete_players_df)} giocatori")

📊 CARICAMENTO DATI FANTACALCIO
Verifica file:
  prices.xlsx: ✅
  stats.xlsx: ✅

Verifica file:
  prices.xlsx: ✅
  stats.xlsx: ✅

✅ File prices.xlsx caricato: (522, 13)
Colonne disponibili:
  1. Quotazioni Fantacalcio Stagione 2025 26
  2. Unnamed: 1
  3. Unnamed: 2
  4. Unnamed: 3
  5. Unnamed: 4
  6. Unnamed: 5
  7. Unnamed: 6
  8. Unnamed: 7
  9. Unnamed: 8
  10. Unnamed: 9
  11. Unnamed: 10
  12. Unnamed: 11
  13. Unnamed: 12

Prime 5 righe di prices.xlsx:
  Quotazioni Fantacalcio Stagione 2025 26 Unnamed: 1 Unnamed: 2   Unnamed: 3  \
0                                      Id          R         RM         Nome   
1                                    2428          P        Por       Sommer   
2                                    5876          P        Por  Di Gregorio   
3                                     572          P        Por        Meret   
4                                    4312          P        Por      Maignan   

  Unnamed: 4 Unnamed: 5 Unnamed: 6 Unnamed: 7 Unnamed: 

In [17]:
# Analisi delle possibili chiavi di matching
print("🔍 ANALISI CHIAVI DI MATCHING")
print("="*50)

# Confronto dei nomi dei giocatori per trovare pattern comuni
if len(prices_df) > 0 and len(complete_players_df) > 0:
    
    # Colonne contenenti nomi nei diversi dataset
    print("📋 COLONNE CONTENENTI NOMI:")
    
    # Prices DataFrame
    name_cols_prices = [col for col in prices_df.columns if 'nome' in col.lower() or 'name' in col.lower() or 'giocatore' in col.lower()]
    print(f"  prices.xlsx: {name_cols_prices}")
    
    # Stats DataFrame  
    if len(stats_df) > 0:
        name_cols_stats = [col for col in stats_df.columns if 'nome' in col.lower() or 'name' in col.lower() or 'giocatore' in col.lower()]
        print(f"  stats.xlsx: {name_cols_stats}")
    
    # SofaScore DataFrame
    name_cols_sofa = [col for col in complete_players_df.columns if 'name' in col.lower()]
    print(f"  SofaScore: {name_cols_sofa}")
    
    # Analizza alcune squadre per vedere se ci sono match
    print(f"\n🏟️ ANALISI SQUADRE:")
    
    # Squadre in prices
    if 'Squadra' in prices_df.columns:
        teams_prices = prices_df['Squadra'].unique()
        print(f"  Squadre in prices ({len(teams_prices)}): {list(teams_prices)[:10]}")
    elif 'squadra' in prices_df.columns:
        teams_prices = prices_df['squadra'].unique() 
        print(f"  Squadre in prices ({len(teams_prices)}): {list(teams_prices)[:10]}")
    
    # Squadre in SofaScore
    teams_sofa = complete_players_df['team_name'].unique()
    print(f"  Squadre in SofaScore ({len(teams_sofa)}): {list(teams_sofa)[:10]}")
    
    # Prova match con nomi giocatori
    print(f"\n👤 TEST MATCHING NOMI GIOCATORI:")
    
    # Prendi i primi 20 nomi da prices per test
    if 'Nome' in prices_df.columns:
        sample_prices_names = prices_df['Nome'].head(20).tolist()
        print(f"  Esempi nomi da prices: {sample_prices_names[:5]}")
        
        # Prendi nomi da SofaScore per confronto
        sample_sofa_names = complete_players_df['name'].head(20).tolist()
        print(f"  Esempi nomi da SofaScore: {sample_sofa_names[:5]}")
        
        # Cerca match esatti
        exact_matches = set(sample_prices_names) & set(complete_players_df['name'].tolist())
        print(f"  Match esatti trovati: {len(exact_matches)}")
        if exact_matches:
            print(f"    Esempi: {list(exact_matches)[:5]}")
    
    # Analizza ruoli/posizioni
    print(f"\n⚽ ANALISI RUOLI/POSIZIONI:")
    
    # Ruoli in prices
    if 'Ruolo' in prices_df.columns:
        roles_prices = prices_df['Ruolo'].unique()
        print(f"  Ruoli in prices: {list(roles_prices)}")
    elif 'R' in prices_df.columns:
        roles_prices = prices_df['R'].unique()
        print(f"  Ruoli in prices: {list(roles_prices)}")
    
    # Ruoli in SofaScore
    roles_sofa = complete_players_df['position'].unique()
    print(f"  Ruoli in SofaScore: {list(roles_sofa)}")

else:
    print("❌ Impossibile analizzare: dati mancanti")

print(f"\n📊 STRATEGIA DI MATCHING PROPOSTA:")
print("1. 🎯 Match primario: Nome giocatore (con normalizzazione)")
print("2. 🏟️ Match secondario: Squadra + Ruolo") 
print("3. 🔧 Fuzzy matching per nomi simili")
print("4. 👥 Verifica manuale per casi dubbi")

🔍 ANALISI CHIAVI DI MATCHING
📋 COLONNE CONTENENTI NOMI:
  prices.xlsx: ['Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12']
  stats.xlsx: ['Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17']
  SofaScore: ['team_name', 'name', 'short_name', 'player_name', 'tournament']

🏟️ ANALISI SQUADRE:
  Squadre in SofaScore (27): ['Inter', 'Parma', 'Monza', 'Milan', 'Como', 'Torino', 'Lecce', 'Juventus', 'Genoa', 'Napoli']

👤 TEST MATCHING NOMI GIOCATORI:

⚽ ANALISI RUOLI/POSIZIONI:
  Ruoli in SofaScore: ['F', 'M', 'D', 'G']

📊 STRATEGIA DI MATCHING PROPOSTA:
1. 🎯 Match primario: Nome giocatore (con normalizzazione)
2. 🏟️ Match secondario: Squadra + Ruolo
3. 🔧 Fuzzy matching per nomi simili


In [18]:
# Analisi dettagliata della struttura dei file Excel
print("🔬 ANALISI DETTAGLIATA STRUTTURA FILE")
print("="*60)

# Analizza prices.xlsx
if len(prices_df) > 0:
    print("\n📊 STRUTTURA prices.xlsx:")
    print(f"Dimensioni: {prices_df.shape}")
    
    # Mostra le prime righe raw per capire la struttura
    print("\nPrime 10 righe (tutte le colonne):")
    print(prices_df.head(10))
    
    # Controlla se ci sono header nelle prime righe
    print("\nAnalisi prime righe per identificare header:")
    for i in range(min(5, len(prices_df))):
        print(f"Riga {i}: {prices_df.iloc[i].tolist()}")

# Analizza stats.xlsx  
if len(stats_df) > 0:
    print("\n📈 STRUTTURA stats.xlsx:")
    print(f"Dimensioni: {stats_df.shape}")
    
    # Mostra le prime righe raw
    print("\nPrime 5 righe (prime 10 colonne):")
    print(stats_df.iloc[:5, :10])
    
    # Analisi prime righe
    print("\nAnalisi prime righe per identificare header:")
    for i in range(min(3, len(stats_df))):
        row_data = stats_df.iloc[i].tolist()[:10]  # Prime 10 colonne
        print(f"Riga {i}: {row_data}")

# Prova a ricaricare con diversi parametri
print(f"\n🔄 TENTATIVO RICARICAMENTO CON HEADER SPECIFICI:")

try:
    # Prova a caricare prices specificando che l'header è nella riga 1 o 2
    prices_test = pd.read_excel("Data/prices.xlsx", header=1)
    print(f"prices.xlsx con header=1: colonne = {list(prices_test.columns)[:10]}")
    
    prices_test2 = pd.read_excel("Data/prices.xlsx", header=0)
    print(f"prices.xlsx con header=0: prime celle = {prices_test2.iloc[0].tolist()[:5]}")
    
except Exception as e:
    print(f"Errore nel test ricaricamento: {e}")

try:
    # Prova a caricare stats con diversi parametri
    stats_test = pd.read_excel("Data/stats.xlsx", header=1) 
    print(f"stats.xlsx con header=1: colonne = {list(stats_test.columns)[:10]}")
    
except Exception as e:
    print(f"Errore nel test stats: {e}")

print(f"\n💡 PROSSIMI PASSI:")
print("1. Identificare la riga corretta degli header")
print("2. Ricaricare i file con parametri corretti")
print("3. Normalizzare i nomi delle colonne")
print("4. Implementare strategia di matching")

🔬 ANALISI DETTAGLIATA STRUTTURA FILE

📊 STRUTTURA prices.xlsx:
Dimensioni: (522, 13)

Prime 10 righe (tutte le colonne):
  Quotazioni Fantacalcio Stagione 2025 26 Unnamed: 1 Unnamed: 2   Unnamed: 3  \
0                                      Id          R         RM         Nome   
1                                    2428          P        Por       Sommer   
2                                    5876          P        Por  Di Gregorio   
3                                     572          P        Por        Meret   
4                                    4312          P        Por      Maignan   
5                                    5841          P        Por       Svilar   
6                                    4431          P        Por  Carnesecchi   
7                                    2521          P        Por       De Gea   
8                                    2814          P        Por     Provedel   
9                                    6966          P        Por        Butez   

In [19]:
# Ricaricamento corretto dei file e implementazione matching
print("🔄 RICARICAMENTO CORRETTO E MATCHING")
print("="*60)

# Funzione per normalizzare i nomi
def normalize_name(name):
    """Normalizza i nomi per il matching"""
    if pd.isna(name) or not isinstance(name, str):
        return ""
    
    # Rimuovi spazi extra, converti in lowercase
    normalized = str(name).strip().lower()
    
    # Rimuovi caratteri speciali comuni
    normalized = normalized.replace("'", "").replace("-", " ").replace(".", "")
    
    # Gestisci casi comuni di nomi diversi
    replacements = {
        "aleksandar": "alexander",
        "aleksandr": "alexander", 
        "nikola": "nicola",
        "luka": "luca",
        "matteo": "mattia",
        "michele": "michael"
    }
    
    for old, new in replacements.items():
        normalized = normalized.replace(old, new)
    
    return normalized

# Funzione per normalizzare nomi squadre
def normalize_team_name(team):
    """Normalizza i nomi delle squadre"""
    if pd.isna(team) or not isinstance(team, str):
        return ""
    
    team_normalized = str(team).strip().lower()
    
    # Mapping squadre comuni
    team_mapping = {
        "inter": "inter",
        "internazionale": "inter", 
        "ac milan": "milan",
        "roma": "roma",
        "juventus": "juventus",
        "juve": "juventus",
        "napoli": "napoli",
        "fiorentina": "fiorentina",
        "atalanta": "atalanta",
        "lazio": "lazio",
        "torino": "torino",
        "genoa": "genoa",
        "bologna": "bologna",
        "lecce": "lecce",
        "udinese": "udinese",
        "sassuolo": "sassuolo",
        "empoli": "empoli",
        "venezia": "venezia",
        "como": "como",
        "parma": "parma",
        "cagliari": "cagliari",
        "hellas verona": "verona",
        "verona": "verona"
    }
    
    return team_mapping.get(team_normalized, team_normalized)

# Ricarica prices.xlsx con header corretto
try:
    # Prova diversi header finché non trova quello giusto
    for header_row in [0, 1, 2]:
        try:
            prices_df_clean = pd.read_excel("Data/prices.xlsx", header=header_row)
            
            # Verifica se ha colonne sensate (non tutte "Unnamed")
            unnamed_cols = [col for col in prices_df_clean.columns if 'Unnamed' in str(col)]
            if len(unnamed_cols) < len(prices_df_clean.columns) * 0.8:  # Meno dell'80% sono "Unnamed"
                print(f"✅ prices.xlsx caricato con header={header_row}")
                print(f"   Colonne: {list(prices_df_clean.columns)}")
                print(f"   Dimensioni: {prices_df_clean.shape}")
                break
        except:
            continue
    else:
        # Se nessun header funziona, usa quello originale
        prices_df_clean = prices_df.copy()
        print("⚠️ Usando file prices originale")
        
except Exception as e:
    prices_df_clean = prices_df.copy()
    print(f"⚠️ Errore ricaricamento prices: {e}")

# Ricarica stats.xlsx con header corretto  
try:
    for header_row in [0, 1, 2]:
        try:
            stats_df_clean = pd.read_excel("Data/stats.xlsx", header=header_row)
            
            unnamed_cols = [col for col in stats_df_clean.columns if 'Unnamed' in str(col)]
            if len(unnamed_cols) < len(stats_df_clean.columns) * 0.8:
                print(f"✅ stats.xlsx caricato con header={header_row}")
                print(f"   Colonne: {list(stats_df_clean.columns)}")
                print(f"   Dimensioni: {stats_df_clean.shape}")
                break
        except:
            continue
    else:
        stats_df_clean = stats_df.copy()
        print("⚠️ Usando file stats originale")
        
except Exception as e:
    stats_df_clean = stats_df.copy()
    print(f"⚠️ Errore ricaricamento stats: {e}")

# Identifica colonne chiave per il matching
print(f"\n🔍 IDENTIFICAZIONE COLONNE CHIAVE:")

# Per prices
prices_name_col = None
prices_team_col = None
prices_role_col = None

for col in prices_df_clean.columns:
    col_lower = str(col).lower()
    if any(word in col_lower for word in ['nome', 'name', 'giocatore', 'player']):
        prices_name_col = col
        break

for col in prices_df_clean.columns:
    col_lower = str(col).lower()
    if any(word in col_lower for word in ['squadra', 'team', 'club']):
        prices_team_col = col
        break
        
for col in prices_df_clean.columns:
    col_lower = str(col).lower()
    if any(word in col_lower for word in ['ruolo', 'role', 'posizione', 'pos', 'r']):
        prices_role_col = col
        break

print(f"  prices.xlsx:")
print(f"    Nome: {prices_name_col}")
print(f"    Squadra: {prices_team_col}")  
print(f"    Ruolo: {prices_role_col}")

# Se non trova colonne, prova con l'analisi dei dati
if not prices_name_col and len(prices_df_clean) > 0:
    # Prova a inferire dalla prima riga di dati
    print(f"\n🔍 ANALISI CONTENUTO per inferire colonne:")
    sample_row = prices_df_clean.iloc[0] if len(prices_df_clean) > 0 else None
    if sample_row is not None:
        print(f"  Prima riga dati: {sample_row.tolist()}")

print(f"\n📊 PROSSIMO: Implementazione algoritmo di matching basato su:")
print("1. Nome giocatore (normalizzato)")
print("2. Squadra (normalizzata)")
print("3. Ruolo/Posizione")
print("4. Fuzzy matching per casi borderline")

🔄 RICARICAMENTO CORRETTO E MATCHING
✅ prices.xlsx caricato con header=1
   Colonne: ['Id', 'R', 'RM', 'Nome', 'Squadra', 'Qt.A', 'Qt.I', 'Diff.', 'Qt.A M', 'Qt.I M', 'Diff.M', 'FVM', 'FVM M']
   Dimensioni: (521, 13)
✅ prices.xlsx caricato con header=1
   Colonne: ['Id', 'R', 'RM', 'Nome', 'Squadra', 'Qt.A', 'Qt.I', 'Diff.', 'Qt.A M', 'Qt.I M', 'Diff.M', 'FVM', 'FVM M']
   Dimensioni: (521, 13)
✅ stats.xlsx caricato con header=1
   Colonne: ['Id', 'R', 'Rm', 'Nome', 'Squadra', 'Pv', 'Mv', 'Fm', 'Gf', 'Gs', 'Rp', 'Rc', 'R+', 'R-', 'Ass', 'Amm', 'Esp', 'Au']
   Dimensioni: (679, 18)

🔍 IDENTIFICAZIONE COLONNE CHIAVE:
  prices.xlsx:
    Nome: Nome
    Squadra: Squadra
    Ruolo: R

📊 PROSSIMO: Implementazione algoritmo di matching basato su:
1. Nome giocatore (normalizzato)
2. Squadra (normalizzata)
3. Ruolo/Posizione
4. Fuzzy matching per casi borderline
✅ stats.xlsx caricato con header=1
   Colonne: ['Id', 'R', 'Rm', 'Nome', 'Squadra', 'Pv', 'Mv', 'Fm', 'Gf', 'Gs', 'Rp', 'Rc', 'R+', 'R-

In [20]:
# Implementazione algoritmo di matching completo
from difflib import SequenceMatcher
import re

print("🤝 IMPLEMENTAZIONE MATCHING ALGORITHM")
print("="*60)

def similarity(a, b):
    """Calcola similarità tra due stringhe"""
    return SequenceMatcher(None, a, b).ratio()

def find_best_match(target_name, target_team, target_role, sofa_df, threshold=0.8):
    """Trova il miglior match nel dataset SofaScore"""
    
    best_match = None
    best_score = 0
    match_details = {}
    
    target_name_norm = normalize_name(target_name)
    target_team_norm = normalize_team_name(target_team)
    
    # Mapping ruoli fantacalcio -> SofaScore
    role_mapping = {
        'P': 'G',    # Portiere
        'D': 'D',    # Difensore  
        'C': 'M',    # Centrocampista
        'A': 'F',    # Attaccante
        'T': 'F'     # Trequartista -> Attaccante
    }
    
    target_role_sofa = role_mapping.get(target_role, target_role)
    
    for idx, row in sofa_df.iterrows():
        sofa_name_norm = normalize_name(row['name'])
        sofa_team_norm = normalize_team_name(row['team_name'])
        sofa_role = row['position']
        
        # Score componenti
        name_score = similarity(target_name_norm, sofa_name_norm)
        team_score = 1.0 if target_team_norm == sofa_team_norm else 0.0
        role_score = 1.0 if target_role_sofa == sofa_role else 0.0
        
        # Score composito (nome ha peso maggiore)
        composite_score = (name_score * 0.6) + (team_score * 0.3) + (role_score * 0.1)
        
        # Bonus per match perfetto di nome
        if name_score > 0.95:
            composite_score += 0.2
            
        # Bonus per match perfetto squadra + ruolo
        if team_score == 1.0 and role_score == 1.0:
            composite_score += 0.1
        
        if composite_score > best_score:
            best_score = composite_score
            best_match = row
            match_details = {
                'name_score': name_score,
                'team_score': team_score, 
                'role_score': role_score,
                'composite_score': composite_score,
                'target_name_norm': target_name_norm,
                'sofa_name_norm': sofa_name_norm,
                'target_team_norm': target_team_norm,
                'sofa_team_norm': sofa_team_norm
            }
    
    if best_score >= threshold:
        return best_match, match_details
    else:
        return None, match_details

# Esegui matching per prices
print("🎯 MATCHING PRICES.XLSX CON SOFASCORE")
print("-" * 40)

matches_prices = []
unmatched_prices = []
match_stats = {
    'exact_matches': 0,
    'good_matches': 0, 
    'fuzzy_matches': 0,
    'no_matches': 0
}

# Pulisci i dati prices
prices_clean = prices_df_clean.dropna(subset=['Nome', 'Squadra', 'R']).copy()
print(f"Giocatori da matchare in prices: {len(prices_clean)}")

for idx, row in prices_clean.iterrows():
    target_name = row['Nome']
    target_team = row['Squadra'] 
    target_role = row['R']
    
    # Prova matching
    match, details = find_best_match(target_name, target_team, target_role, complete_players_df)
    
    if match is not None:
        # Categorizza il tipo di match
        score = details['composite_score']
        if score >= 0.95:
            match_type = 'exact'
            match_stats['exact_matches'] += 1
        elif score >= 0.85:
            match_type = 'good'
            match_stats['good_matches'] += 1
        else:
            match_type = 'fuzzy'
            match_stats['fuzzy_matches'] += 1
            
        # Combina i dati
        combined_row = {
            # Dati fantacalcio (prioritari)
            'id_fantacalcio': row['Id'],
            'nome': target_name,
            'squadra': target_team,
            'ruolo': target_role,
            'quota_iniziale': row['Qt.I'],
            'quota_attuale': row['Qt.A'],
            'differenza_quota': row['Diff.'],
            'fvm': row['FVM'],
            
            # Dati SofaScore
            'player_id_sofa': match['player_id'],
            'nome_sofa': match['name'],
            'squadra_sofa': match['team_name'],
            'ruolo_sofa': match['position'],
            'goals': match.get('goals', 0),
            'assists': match.get('assists', 0),
            'appearances': match.get('appearances', 0),
            'minutes_played': match.get('minutes_played', 0),
            'rating': match.get('rating', 0),
            'expected_goals': match.get('expected_goals', 0),
            'expected_assists': match.get('expected_assists', 0),
            'shots_on_target': match.get('shots_on_target', 0),
            'total_shots': match.get('total_shots', 0),
            'accurate_passes': match.get('accurate_passes', 0),
            'total_passes': match.get('total_passes', 0),
            'successful_dribbles': match.get('successful_dribbles', 0),
            'tackles': match.get('tackles', 0),
            'interceptions': match.get('interceptions', 0),
            'yellow_cards': match.get('yellow_cards', 0),
            'red_cards': match.get('red_cards', 0),
            
            # Metadati matching
            'match_type': match_type,
            'match_score': score,
            'name_similarity': details['name_score']
        }
        
        matches_prices.append(combined_row)
    else:
        match_stats['no_matches'] += 1
        unmatched_prices.append({
            'nome': target_name,
            'squadra': target_team,
            'ruolo': target_role,
            'best_score': details.get('composite_score', 0)
        })

# Risultati matching
print(f"\n📊 RISULTATI MATCHING PRICES:")
print(f"  ✅ Match esatti: {match_stats['exact_matches']}")
print(f"  ✅ Match buoni: {match_stats['good_matches']}")
print(f"  ⚠️ Match fuzzy: {match_stats['fuzzy_matches']}")
print(f"  ❌ Non matchati: {match_stats['no_matches']}")
print(f"  📈 Tasso successo: {((match_stats['exact_matches'] + match_stats['good_matches'] + match_stats['fuzzy_matches']) / len(prices_clean) * 100):.1f}%")

# Crea DataFrame finale
fantacalcio_complete_df = pd.DataFrame(matches_prices)
print(f"\n✅ DataFrame completo creato: {fantacalcio_complete_df.shape}")

# Mostra alcuni esempi di match
print(f"\n👥 ESEMPI DI MATCH RIUSCITI:")
if len(fantacalcio_complete_df) > 0:
    sample_matches = fantacalcio_complete_df.head(10)[['nome', 'squadra', 'ruolo', 'nome_sofa', 'squadra_sofa', 'goals', 'assists', 'rating', 'match_type', 'match_score']].round(2)
    print(sample_matches.to_string(index=False))

# Mostra non matchati
if len(unmatched_prices) > 0:
    print(f"\n❌ ESEMPI NON MATCHATI:")
    unmatched_df = pd.DataFrame(unmatched_prices)
    print(unmatched_df.head(10).to_string(index=False))

🤝 IMPLEMENTAZIONE MATCHING ALGORITHM
🎯 MATCHING PRICES.XLSX CON SOFASCORE
----------------------------------------
Giocatori da matchare in prices: 521

📊 RISULTATI MATCHING PRICES:
  ✅ Match esatti: 48
  ✅ Match buoni: 238
  ⚠️ Match fuzzy: 68
  ❌ Non matchati: 167
  📈 Tasso successo: 67.9%

✅ DataFrame completo creato: (354, 31)

👥 ESEMPI DI MATCH RIUSCITI:
       nome    squadra ruolo           nome_sofa squadra_sofa  goals  assists  rating match_type  match_score
     Sommer      Inter     P         Yann Sommer        Inter    0.0      0.0    7.09       good         0.92
Di Gregorio   Juventus     P Michele Di Gregorio     Juventus    0.0      0.0    7.01       good         0.94
      Meret     Napoli     P          Alex Meret       Napoli    0.0      0.0    6.97       good         0.90
    Maignan      Milan     P        Mike Maignan        Milan    0.0      0.0    7.08       good         0.94
     Svilar       Roma     P         Mile Svilar         Roma    0.0      0.0    7.19   

In [21]:
# Matching completo tra i dataset: prices (base), stats (via Id), SofaScore (via nome)
def create_complete_fantacalcio_dataset():
    """Crea il dataset completo combinando prices (base), stats (via Id) e SofaScore (via nome)"""
    
    print("🔗 CREAZIONE DATASET COMPLETO FANTACALCIO")
    print("="*60)
    
    # 1. MERGE PRICES + STATS (tramite colonna Id)
    print("1️⃣ Merge prices + stats tramite colonna 'Id'...")
    
    # Assicuriamoci che le colonne Id siano dello stesso tipo
    prices_df_clean['Id'] = prices_df_clean['Id'].astype(str)
    stats_df_clean['Id'] = stats_df_clean['Id'].astype(str)
    
    # Merge prices (base) con stats
    fantacalcio_base = prices_df_clean.merge(
        stats_df_clean, 
        on='Id', 
        how='left',
        suffixes=('_prices', '_stats')
    )
    
    print(f"   Prices: {len(prices_df_clean)} giocatori")
    print(f"   Stats: {len(stats_df_clean)} giocatori")
    print(f"   Merge result: {len(fantacalcio_base)} giocatori")
    
    # Conta quanti hanno match con stats
    stats_columns = ['Pv', 'Mv', 'Fm', 'Gf', 'Gs']  # Colonne che vengono solo da stats
    matched_stats = fantacalcio_base['Pv'].notna().sum()
    print(f"   Match con stats: {matched_stats} giocatori ({matched_stats/len(fantacalcio_base)*100:.1f}%)")
    
    # 2. FUZZY MATCHING CON SOFASCORE (threshold 0.6)
    print("\n2️⃣ Fuzzy matching con SofaScore (threshold 0.6)...")
    
    # Prepara i dati SofaScore per il matching
    sofa_for_matching = complete_players_df[complete_players_df['goals'].notna()].copy()
    sofa_for_matching['name_clean'] = sofa_for_matching['name'].str.lower().str.strip()
    sofa_for_matching['team_clean'] = sofa_for_matching['team_name'].str.lower().str.strip()
    
    # Prepara i dati fantacalcio per il matching - USA I NOMI CORRETTI DELLE COLONNE
    fantacalcio_base['name_clean'] = fantacalcio_base['Nome_prices'].str.lower().str.strip()
    fantacalcio_base['team_clean'] = fantacalcio_base['Squadra_prices'].str.lower().str.strip()
    
    print(f"   SofaScore giocatori disponibili: {len(sofa_for_matching)}")
    print(f"   Fantacalcio giocatori: {len(fantacalcio_base)}")
    
    # Esegui fuzzy matching con threshold 0.6
    threshold = 0.6
    matches_sofa = []
    unmatched_fantacalcio = []
    
    for idx, fanta_player in fantacalcio_base.iterrows():
        best_match = None
        best_score = 0
        
        target_name = fanta_player['name_clean']
        target_team = fanta_player['team_clean']
        target_role = fanta_player['R_prices']  # Ruolo corretto
        
        for sofa_idx, sofa_player in sofa_for_matching.iterrows():
            # Score nome (peso maggiore)
            name_score = fuzz.ratio(target_name, sofa_player['name_clean']) / 100
            
            # Score squadra (peso medio)
            team_score = fuzz.ratio(target_team, sofa_player['team_clean']) / 100
            
            # Bonus ruolo compatibile (peso minore)
            role_bonus = 0
            if target_role and sofa_player['position']:
                role_mapping = {'P': 'G', 'D': 'D', 'C': 'M', 'A': 'F'}
                if role_mapping.get(target_role) == sofa_player['position']:
                    role_bonus = 0.1
            
            # Score finale pesato (nome 70%, squadra 25%, ruolo 5%)
            final_score = (name_score * 0.70) + (team_score * 0.25) + role_bonus
            
            if final_score > best_score and final_score >= threshold:
                best_score = final_score
                best_match = sofa_player
        
        if best_match is not None:
            match_info = {
                'fantacalcio_idx': idx,
                'sofa_idx': best_match.name,
                'score': best_score,
                'fanta_name': fanta_player['Nome_prices'],
                'sofa_name': best_match['name'],
                'fanta_team': fanta_player['Squadra_prices'],
                'sofa_team': best_match['team_name'],
                'match_type': 'fuzzy_match'
            }
            matches_sofa.append(match_info)
        else:
            unmatched_fantacalcio.append({
                'idx': idx,
                'name': fanta_player['Nome_prices'],
                'team': fanta_player['Squadra_prices'],
                'role': fanta_player['R_prices']
            })
    
    print(f"   Match trovati: {len(matches_sofa)} ({len(matches_sofa)/len(fantacalcio_base)*100:.1f}%)")
    print(f"   Non matched: {len(unmatched_fantacalcio)} ({len(unmatched_fantacalcio)/len(fantacalcio_base)*100:.1f}%)")
    
    # 3. CREAZIONE DATASET FINALE
    print("\n3️⃣ Creazione dataset finale...")
    
    # Inizializza il dataset finale con i dati fantacalcio
    final_dataset = fantacalcio_base.copy()
    
    # Aggiungi colonne SofaScore (inizializzate a NaN)
    sofa_columns = [
        'sofa_goals', 'sofa_assists', 'sofa_appearances', 'sofa_minutes_played',
        'sofa_rating', 'sofa_expected_goals', 'sofa_expected_assists',
        'sofa_shots_on_target', 'sofa_total_shots', 'sofa_successful_dribbles',
        'sofa_accurate_crosses', 'sofa_pass_accuracy', 'sofa_tackles',
        'sofa_interceptions', 'sofa_yellow_cards', 'sofa_red_cards',
        'sofa_match_score', 'sofa_match_type'
    ]
    
    for col in sofa_columns:
        final_dataset[col] = None
    
    # Popola i dati SofaScore per i match trovati
    for match in matches_sofa:
        fanta_idx = match['fantacalcio_idx']
        sofa_data = sofa_for_matching.loc[match['sofa_idx']]
        
        final_dataset.loc[fanta_idx, 'sofa_goals'] = sofa_data['goals']
        final_dataset.loc[fanta_idx, 'sofa_assists'] = sofa_data['assists']
        final_dataset.loc[fanta_idx, 'sofa_appearances'] = sofa_data['appearances']
        final_dataset.loc[fanta_idx, 'sofa_minutes_played'] = sofa_data['minutes_played']
        final_dataset.loc[fanta_idx, 'sofa_rating'] = sofa_data['rating']
        final_dataset.loc[fanta_idx, 'sofa_expected_goals'] = sofa_data['expected_goals']
        final_dataset.loc[fanta_idx, 'sofa_expected_assists'] = sofa_data['expected_assists']
        final_dataset.loc[fanta_idx, 'sofa_shots_on_target'] = sofa_data['shots_on_target']
        final_dataset.loc[fanta_idx, 'sofa_total_shots'] = sofa_data['total_shots']
        final_dataset.loc[fanta_idx, 'sofa_successful_dribbles'] = sofa_data['successful_dribbles']
        final_dataset.loc[fanta_idx, 'sofa_accurate_crosses'] = sofa_data['accurate_crosses']
        final_dataset.loc[fanta_idx, 'sofa_pass_accuracy'] = sofa_data['pass_accuracy']
        final_dataset.loc[fanta_idx, 'sofa_tackles'] = sofa_data['tackles']
        final_dataset.loc[fanta_idx, 'sofa_interceptions'] = sofa_data['interceptions']
        final_dataset.loc[fanta_idx, 'sofa_yellow_cards'] = sofa_data['yellow_cards']
        final_dataset.loc[fanta_idx, 'sofa_red_cards'] = sofa_data['red_cards']
        final_dataset.loc[fanta_idx, 'sofa_match_score'] = match['score']
        final_dataset.loc[fanta_idx, 'sofa_match_type'] = match['match_type']
    
    # 4. STATISTICHE FINALI
    print("\n📊 STATISTICHE DATASET FINALE:")
    print(f"   Totale giocatori: {len(final_dataset)}")
    print(f"   Con dati SofaScore: {final_dataset['sofa_goals'].notna().sum()} ({final_dataset['sofa_goals'].notna().sum()/len(final_dataset)*100:.1f}%)")
    print(f"   Con statistiche fantacalcio: {final_dataset['Pv'].notna().sum()} ({final_dataset['Pv'].notna().sum()/len(final_dataset)*100:.1f}%)")
    
    # Distribuzione per ruolo
    if 'R_prices' in final_dataset.columns:
        role_dist = final_dataset['R_prices'].value_counts()
        print(f"\n   Distribuzione per ruolo:")
        for role, count in role_dist.items():
            sofa_count = final_dataset[final_dataset['R_prices'] == role]['sofa_goals'].notna().sum()
            print(f"     {role}: {count} tot, {sofa_count} con SofaScore ({sofa_count/count*100:.1f}%)")
    
    return final_dataset, matches_sofa, unmatched_fantacalcio

# Esegui il matching completo
fantacalcio_complete_df, matches_details, unmatched_list = create_complete_fantacalcio_dataset()

🔗 CREAZIONE DATASET COMPLETO FANTACALCIO
1️⃣ Merge prices + stats tramite colonna 'Id'...
   Prices: 521 giocatori
   Stats: 679 giocatori
   Merge result: 521 giocatori
   Match con stats: 387 giocatori (74.3%)

2️⃣ Fuzzy matching con SofaScore (threshold 0.6)...
   SofaScore giocatori disponibili: 543
   Fantacalcio giocatori: 521


NameError: name 'fuzz' is not defined

In [ ]:
# Analisi dei risultati del matching
print("🔍 ANALISI DETTAGLIATA DEL MATCHING")
print("="*60)

# 1. Esempi di match trovati
print("🎯 ESEMPI DI MATCH TROVATI (top 10 per score):")
matches_df = pd.DataFrame(matches_details)
if len(matches_df) > 0:
    top_matches = matches_df.nlargest(10, 'score')
    for _, match in top_matches.iterrows():
        print(f"   {match['fanta_name']} ({match['fanta_team']}) -> {match['sofa_name']} ({match['sofa_team']}) | Score: {match['score']:.3f}")

# 2. Esempi di giocatori non matchati
print(f"\n❌ GIOCATORI NON MATCHATI (primi 10 di {len(unmatched_list)}):")
for player in unmatched_list[:10]:
    print(f"   {player['name']} ({player['team']}) - Ruolo: {player['role']}")

# 3. Confronto dati integrati
print(f"\n📊 ESEMPI DATASET INTEGRATO:")
print("Giocatori con tutti i dati disponibili (primi 5):")

# Seleziona giocatori che hanno sia dati fantacalcio che SofaScore
complete_data = fantacalcio_complete_df[
    (fantacalcio_complete_df['Pv'].notna()) & 
    (fantacalcio_complete_df['sofa_goals'].notna())
].head(5)

display_columns = [
    'Nome_prices', 'Squadra_prices', 'R_prices', 'Qt.A',  # Fantacalcio base
    'Pv', 'Mv', 'Gf', 'Gs',  # Stats fantacalcio
    'sofa_goals', 'sofa_assists', 'sofa_rating', 'sofa_appearances'  # SofaScore
]

print(complete_data[display_columns].to_string(index=False))

# 4. Statistiche di copertura dati
print(f"\n📈 COPERTURA DATI PER RUOLO:")
for role in ['P', 'D', 'C', 'A']:
    role_data = fantacalcio_complete_df[fantacalcio_complete_df['R_prices'] == role]
    total = len(role_data)
    with_fanta_stats = role_data['Pv'].notna().sum()
    with_sofa = role_data['sofa_goals'].notna().sum()
    with_both = role_data[(role_data['Pv'].notna()) & (role_data['sofa_goals'].notna())].shape[0]
    
    print(f"   {role}: {total} tot | Fanta: {with_fanta_stats} ({with_fanta_stats/total*100:.1f}%) | SofaScore: {with_sofa} ({with_sofa/total*100:.1f}%) | Entrambi: {with_both} ({with_both/total*100:.1f}%)")

# 5. Distribuzione score del matching
if len(matches_df) > 0:
    print(f"\n📊 DISTRIBUZIONE SCORE MATCHING:")
    score_ranges = [
        (0.9, 1.0, "Ottimo"),
        (0.8, 0.9, "Buono"),
        (0.7, 0.8, "Discreto"),
        (0.6, 0.7, "Accettabile")
    ]
    
    for min_score, max_score, label in score_ranges:
        count = matches_df[(matches_df['score'] >= min_score) & (matches_df['score'] < max_score)].shape[0]
        print(f"   {label} ({min_score:.1f}-{max_score:.1f}): {count} match ({count/len(matches_df)*100:.1f}%)")

print(f"\n✅ Dataset finale creato: {len(fantacalcio_complete_df)} giocatori totali")

🔍 ANALISI DETTAGLIATA DEL MATCHING
🎯 ESEMPI DI MATCH TROVATI (top 10 per score):
   Wesley (Roma) -> Wesley (Roma) | Score: 1.050
   Bremer (Juventus) -> Bremer (Juventus) | Score: 1.050
   Carlos Augusto (Inter) -> Carlos Augusto (Inter) | Score: 1.050
   Patric (Lazio) -> Patric (Lazio) | Score: 1.050
   Juan Jesus (Napoli) -> Juan Jesus (Napoli) | Score: 1.050
   Tiago Gabriel (Lecce) -> Tiago Gabriel (Lecce) | Score: 1.050
   Tete Morente (Lecce) -> Tete Morente (Lecce) | Score: 1.050
   Hernani (Parma) -> Hernani (Parma) | Score: 1.050
   Sergi Roberto (Como) -> Sergi Roberto (Como) | Score: 1.050
   Pedro (Lazio) -> Pedro (Lazio) | Score: 1.050

❌ GIOCATORI NON MATCHATI (primi 10 di 53):
   Israel (Torino) - Ruolo: P
   Turati (Sassuolo) - Ruolo: P
   Muric (Sassuolo) - Ruolo: P
   Pessina Mas. (Bologna) - Ruolo: P
   Sommariva (Genoa) - Ruolo: P
   Furlanetto (Lazio) - Ruolo: P
   Fruchtl (Lecce) - Ruolo: P
   Samooja (Lecce) - Ruolo: P
   Torriani (Milan) - Ruolo: P
   Corvi (P

In [ ]:
# Salvataggio del dataset finale integrato
def save_integrated_dataset():
    """Salva il dataset finale integrato in diversi formati"""
    
    print("💾 SALVATAGGIO DATASET INTEGRATO")
    print("="*50)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    try:
        # 1. Dataset completo in Excel (più leggibile)
        excel_file = f"prices_ext.xlsx"
        
        # Riordina le colonne per essere più leggibili
        ordered_columns = [
            # Dati base
            'Id', 'Nome_prices', 'Squadra_prices', 'R_prices', 'RM_prices',
            # Quotazioni
            'Qt.A', 'Qt.I', 'Diff.', 'FVM',
            # Statistiche fantacalcio
            'Pv', 'Mv', 'Fm', 'Gf', 'Gs', 'Rp', 'Rc', 'Ass', 'Amm', 'Esp',
            # Statistiche SofaScore
            'sofa_goals', 'sofa_assists', 'sofa_appearances', 'sofa_minutes_played',
            'sofa_rating', 'sofa_expected_goals', 'sofa_expected_assists',
            'sofa_shots_on_target', 'sofa_total_shots', 'sofa_successful_dribbles',
            'sofa_accurate_crosses', 'sofa_pass_accuracy', 'sofa_tackles',
            'sofa_interceptions', 'sofa_yellow_cards', 'sofa_red_cards',
            'sofa_match_score', 'sofa_match_type'
        ]
        
        # Seleziona solo le colonne che esistono
        available_columns = [col for col in ordered_columns if col in fantacalcio_complete_df.columns]
        dataset_to_save = fantacalcio_complete_df[available_columns].copy()
        
        dataset_to_save.to_excel(excel_file, index=False)
        print(f"✅ Excel salvato: {excel_file}")
        
        # 2. CSV per analisi con pandas
        csv_file = f"prices_ext.csv"
        dataset_to_save.to_csv(csv_file, index=False, encoding='utf-8')
        print(f"✅ CSV salvato: {csv_file}")
        
        # 3. Solo giocatori con dati completi
        complete_players = fantacalcio_complete_df[
            (fantacalcio_complete_df['Pv'].notna()) & 
            (fantacalcio_complete_df['sofa_goals'].notna())
        ][available_columns].copy()
        
        complete_excel = f"prices_ext_compl.xlsx"
        complete_players.to_excel(complete_excel, index=False)
        print(f"✅ Dataset completo salvato: {complete_excel} ({len(complete_players)} giocatori)")
        
        # 4. File di metadati
        metadata = {
            'timestamp': datetime.now().isoformat(),
            'total_players': len(fantacalcio_complete_df),
            'players_with_fantacalcio_stats': fantacalcio_complete_df['Pv'].notna().sum(),
            'players_with_sofascore_data': fantacalcio_complete_df['sofa_goals'].notna().sum(),
            'players_with_complete_data': len(complete_players),
            'matching_threshold': 0.6,
            'match_success_rate': len(matches_details) / len(fantacalcio_complete_df),
            'files_created': [excel_file, csv_file, complete_excel],
            'columns_included': available_columns
        }
        
        metadata_file = f"fantacalcio_metadata_{timestamp}.json"
        with open(metadata_file, 'w', encoding='utf-8') as f:
            json.dump(metadata, f, indent=2, ensure_ascii=False)
        print(f"✅ Metadati salvati: {metadata_file}")
        
        # 5. Statistiche finali
        print(f"\n📊 RIEPILOGO FINALE:")
        print(f"   Giocatori totali: {len(fantacalcio_complete_df)}")
        print(f"   Con quotazioni: {len(fantacalcio_complete_df)} (100.0%)")
        print(f"   Con stats fantacalcio: {fantacalcio_complete_df['Pv'].notna().sum()} ({fantacalcio_complete_df['Pv'].notna().sum()/len(fantacalcio_complete_df)*100:.1f}%)")
        print(f"   Con dati SofaScore: {fantacalcio_complete_df['sofa_goals'].notna().sum()} ({fantacalcio_complete_df['sofa_goals'].notna().sum()/len(fantacalcio_complete_df)*100:.1f}%)")
        print(f"   Con dati completi: {len(complete_players)} ({len(complete_players)/len(fantacalcio_complete_df)*100:.1f}%)")
        
        return [excel_file, csv_file, complete_excel, metadata_file]
        
    except Exception as e:
        print(f"❌ Errore nel salvataggio: {e}")
        return []

# Esegui il salvataggio
saved_files = save_integrated_dataset()

print(f"\n🎉 PROCESSO COMPLETATO!")
print("Dataset fantacalcio integrato con successo creato e salvato.")

💾 SALVATAGGIO DATASET INTEGRATO
✅ Excel salvato: prices_ext.xlsx
✅ CSV salvato: prices_ext.csv
✅ Excel salvato: prices_ext.xlsx
✅ CSV salvato: prices_ext.csv
✅ Dataset completo salvato: prices_ext_compl.xlsx (364 giocatori)
❌ Errore nel salvataggio: Object of type int64 is not JSON serializable

🎉 PROCESSO COMPLETATO!
Dataset fantacalcio integrato con successo creato e salvato.
✅ Dataset completo salvato: prices_ext_compl.xlsx (364 giocatori)
❌ Errore nel salvataggio: Object of type int64 is not JSON serializable

🎉 PROCESSO COMPLETATO!
Dataset fantacalcio integrato con successo creato e salvato.


In [ ]:
# Import necessari per il fuzzy matching
try:
    from fuzzywuzzy import fuzz
    print("✅ fuzzywuzzy disponibile")
except ImportError:
    print("⚠️ Installazione fuzzywuzzy...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'fuzzywuzzy', 'python-Levenshtein'])
    from fuzzywuzzy import fuzz
    print("✅ fuzzywuzzy installato e importato")

⚠️ Installazione fuzzywuzzy...
✅ fuzzywuzzy installato e importato


In [ ]:
# Debug: verifica colonne disponibili nei DataFrame
print("🔍 VERIFICA COLONNE DISPONIBILI:")
print("="*50)

print("Colonne prices_df_clean:")
print(list(prices_df_clean.columns))
print(f"Shape: {prices_df_clean.shape}")

print("\nColonne stats_df_clean:")
print(list(stats_df_clean.columns))
print(f"Shape: {stats_df_clean.shape}")

print("\nColonne complete_players_df:")
print(list(complete_players_df.columns))
print(f"Shape: {complete_players_df.shape}")

print("\nPrime 3 righe prices_df_clean:")
print(prices_df_clean.head(3))

print("\nPrime 3 righe stats_df_clean:")
print(stats_df_clean.head(3))

🔍 VERIFICA COLONNE DISPONIBILI:
Colonne prices_df_clean:
['Id', 'R', 'RM', 'Nome', 'Squadra', 'Qt.A', 'Qt.I', 'Diff.', 'Qt.A M', 'Qt.I M', 'Diff.M', 'FVM', 'FVM M']
Shape: (519, 13)

Colonne stats_df_clean:
['Id', 'R', 'Rm', 'Nome', 'Squadra', 'Pv', 'Mv', 'Fm', 'Gf', 'Gs', 'Rp', 'Rc', 'R+', 'R-', 'Ass', 'Amm', 'Esp', 'Au']
Shape: (679, 18)

Colonne complete_players_df:
['player_id', 'team_id', 'team_name', 'name', 'short_name', 'slug', 'position', 'jersey_number', 'height', 'date_of_birth', 'preferred_foot', 'market_value', 'nationality', 'player_name', 'season', 'tournament', 'goals', 'assists', 'goals_assists_sum', 'expected_goals', 'expected_assists', 'big_chances_created', 'big_chances_missed', 'shots_on_target', 'total_shots', 'shots_from_inside_box', 'key_passes', 'pass_to_assist', 'appearances', 'minutes_played', 'rating', 'total_rating', 'count_rating', 'accurate_passes', 'total_passes', 'accurate_passes_percentage', 'accurate_long_balls', 'total_long_balls', 'accurate_crosses'

In [ ]:
# Matching migliorato con threshold 0.6 e collegamento prices-stats via Id
def improved_matching_with_stats():
    """Matching migliorato che collega prices, stats e SofaScore con threshold più bassa"""
    
    print("🔄 MATCHING MIGLIORATO CON THRESHOLD 0.6")
    print("="*50)
    
    # Prima collega prices e stats tramite la colonna Id
    print("1️⃣ Collegamento prices-stats tramite colonna Id...")
    
    if 'Id' in prices_df_clean.columns and 'Id' in stats_df_clean.columns:
        # Merge di prices e stats tramite Id
        fantacalcio_base = prices_df_clean.merge(
            stats_df_clean,
            on='Id',
            how='left',
            suffixes=('_prices', '_stats')
        )
        
        print(f"   ✅ Collegati {len(fantacalcio_base)} giocatori tra prices e stats")
        
        # Usa il nome da prices (più pulito) se disponibile, altrimenti da stats
        if 'Nome_prices' in fantacalcio_base.columns and 'Nome_stats' in fantacalcio_base.columns:
            fantacalcio_base['Nome'] = fantacalcio_base['Nome_prices'].fillna(fantacalcio_base['Nome_stats'])
        elif 'Nome_prices' in fantacalcio_base.columns:
            fantacalcio_base['Nome'] = fantacalcio_base['Nome_prices']
        elif 'Nome_stats' in fantacalcio_base.columns:
            fantacalcio_base['Nome'] = fantacalcio_base['Nome_stats']
            
        # Stesso per altre colonne importanti
        if 'Ruolo_prices' in fantacalcio_base.columns and 'Ruolo_stats' in fantacalcio_base.columns:
            fantacalcio_base['Ruolo'] = fantacalcio_base['Ruolo_prices'].fillna(fantacalcio_base['Ruolo_stats'])
        elif 'Ruolo_prices' in fantacalcio_base.columns:
            fantacalcio_base['Ruolo'] = fantacalcio_base['Ruolo_prices']
            
        if 'Squadra_prices' in fantacalcio_base.columns and 'Squadra_stats' in fantacalcio_base.columns:
            fantacalcio_base['Squadra'] = fantacalcio_base['Squadra_prices'].fillna(fantacalcio_base['Squadra_stats'])
        elif 'Squadra_prices' in fantacalcio_base.columns:
            fantacalcio_base['Squadra'] = fantacalcio_base['Squadra_prices']
            
    else:
        print("   ⚠️ Colonna Id non trovata, uso solo prices")
        fantacalcio_base = prices_df_clean.copy()
        fantacalcio_base['Nome'] = fantacalcio_base[prices_name_col]
        fantacalcio_base['Ruolo'] = fantacalcio_base[prices_role_col]
        fantacalcio_base['Squadra'] = fantacalcio_base[prices_team_col]
    
    print(f"   Dataset fantacalcio base: {len(fantacalcio_base)} giocatori")
    
    # 2. Ora fai il matching con SofaScore usando threshold 0.6
    print("\n2️⃣ Matching con SofaScore (threshold 0.6)...")
    
    from difflib import SequenceMatcher
    
    def similarity_score(a, b):
        """Calcola similarità tra due stringhe"""
        if pd.isna(a) or pd.isna(b):
            return 0
        return SequenceMatcher(None, str(a).lower().strip(), str(b).lower().strip()).ratio()
    
    def normalize_team_name(team):
        """Normalizza nomi squadre"""
        if pd.isna(team):
            return ""
        team = str(team).lower().strip()
        
        # Mapping squadre comuni
        team_mapping = {
            'atalanta': 'atalanta',
            'bologna': 'bologna',
            'cagliari': 'cagliari',
            'como': 'como',
            'empoli': 'empoli',
            'fiorentina': 'fiorentina',
            'genoa': 'genoa',
            'hellas verona': 'verona',
            'verona': 'verona',
            'inter': 'inter',
            'juventus': 'juventus',
            'lazio': 'lazio',
            'lecce': 'lecce',
            'milan': 'milan',
            'napoli': 'napoli',
            'parma': 'parma',
            'roma': 'roma',
            'torino': 'torino',
            'udinese': 'udinese',
            'venezia': 'venezia'
        }
        
        return team_mapping.get(team, team)
    
    # Prepara dati SofaScore per matching
    sofa_players = complete_players_df[
        ['name', 'team_name', 'position', 'goals', 'assists', 'rating', 'minutes_played', 'appearances']
    ].copy()
    
    # Normalizza nomi squadre
    fantacalcio_base['squadra_norm'] = fantacalcio_base['Squadra'].apply(normalize_team_name)
    sofa_players['team_norm'] = sofa_players['team_name'].apply(normalize_team_name)
    
    matches_improved = []
    matched_indices = set()
    
    for idx, row in fantacalcio_base.iterrows():
        target_name = str(row['Nome']).strip()
        target_team = row['squadra_norm']
        target_role = str(row['Ruolo']).strip() if pd.notna(row['Ruolo']) else ""
        
        best_match = None
        best_score = 0
        
        for sofa_idx, sofa_row in sofa_players.iterrows():
            if sofa_idx in matched_indices:
                continue
                
            sofa_name = str(sofa_row['name']).strip()
            sofa_team = sofa_row['team_norm']
            
            # Calcola score nome
            name_score = similarity_score(target_name, sofa_name)
            
            # Bonus se squadra corrisponde
            team_bonus = 0.15 if target_team == sofa_team else 0
            
            # Score totale
            total_score = name_score + team_bonus
            
            # Soglia più bassa: 0.6
            if total_score >= 0.6 and total_score > best_score:
                best_score = total_score
                best_match = {
                    'fantacalcio_idx': idx,
                    'sofa_idx': sofa_idx,
                    'name_fantacalcio': target_name,
                    'name_sofa': sofa_name,
                    'team_fantacalcio': target_team,
                    'team_sofa': sofa_team,
                    'score': total_score,
                    'goals': sofa_row['goals'],
                    'assists': sofa_row['assists'],
                    'rating': sofa_row['rating'],
                    'minutes': sofa_row['minutes_played'],
                    'appearances': sofa_row['appearances']
                }
        
        if best_match:
            matches_improved.append(best_match)
            matched_indices.add(best_match['sofa_idx'])
    
    print(f"   ✅ Trovati {len(matches_improved)} matches con threshold 0.6")
    
    # 3. Crea dataset finale combinato
    print("\n3️⃣ Creazione dataset finale...")
    
    # Inizia con il dataset fantacalcio base
    final_dataset = fantacalcio_base.copy()
    
    # Aggiungi colonne SofaScore
    sofa_columns = ['sofa_goals', 'sofa_assists', 'sofa_rating', 'sofa_minutes', 'sofa_appearances', 'sofa_match_score']
    for col in sofa_columns:
        final_dataset[col] = None
    
    # Popola i dati SofaScore per i match trovati
    for match in matches_improved:
        idx = match['fantacalcio_idx']
        final_dataset.loc[idx, 'sofa_goals'] = match['goals']
        final_dataset.loc[idx, 'sofa_assists'] = match['assists']
        final_dataset.loc[idx, 'sofa_rating'] = match['rating']
        final_dataset.loc[idx, 'sofa_minutes'] = match['minutes']
        final_dataset.loc[idx, 'sofa_appearances'] = match['appearances']
        final_dataset.loc[idx, 'sofa_match_score'] = match['score']
    
    print(f"   ✅ Dataset finale: {len(final_dataset)} giocatori")
    print(f"   📊 Con dati SofaScore: {len([m for m in matches_improved])} giocatori")
    print(f"   📊 Match rate: {len(matches_improved)/len(final_dataset)*100:.1f}%")
    
    return final_dataset, matches_improved

# Esegui il matching migliorato
fantacalcio_complete_improved, matches_detailed = improved_matching_with_stats()

print(f"\n🎉 MATCHING COMPLETATO!")
print(f"📈 Miglioramento: da {len(fantacalcio_complete_df)} a {len(fantacalcio_complete_improved)} giocatori nel dataset finale")